# 入門1〜4 測定の記録を解析する（ノートブック版）

入門1〜4 は Pico・テスタ・サーボを手で動かすのが中心ですが、各章の最後には、測った値を Mac の Python で解析する部分があります。このノートブックは、その**解析の部分だけ**を、本文のファイルと同じ形の**合成データ**で試せるように1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約1分）。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 1 | LED の抵抗・PWM・ガンマ補正の数値例 | 入門1-3 | （計算だけ） |
| 2 | LED の電圧・電流の解析（キルヒホッフ、理想係数 $n$、GPIO の出力抵抗、電流計の負担電圧） | 入門1 手順16 | `led_iv.py`、`led_iv.csv` |
| 3 | チャタリングの継続時間（Wilson 区間、順序統計量の分位点の区間、疑似反復、検定） | 入門1 段階5 | `bounce_analysis.py` |
| 4 | はんだ付けの検査（ゼロ点、マスキング、ICC、I2C の失敗率） | 入門2 手順25 | `joint_analysis.py`、`joints.csv` |
| 5 | I2C の立ち上がり時間・INA219 の校正値・有効サンプル数の数値例、BME280 の補正式の検算 | 入門3-3、手順2 | `bme280.py`、`ina219.py`、`test_bme280_vector.py` |
| 6 | 静止状態の雑音（自己相関、HAC、アラン偏差）と PWM の電流 | 入門3 の雑音の手順 | `analyze_noise.py` |
| 7 | BME280 と Pico の内蔵温度センサの比較（回帰、ダービン・ワトソン比、相互相関） | 入門3 の温度の比較 | `analyze_env.py` |
| 8 | ロジックアナライザの記録からパルス幅と周期 | 入門4 手順11 | `la_pulse.py` |
| 9 | サーボの校正（回帰、ヒステリシス、逆推定、F 検定） | 入門4 手順14〜16 | `servo_analysis.py` |

- 教科書の `~/home-lab/` の代わりに、このノートブックのフォルダに `intro1/`〜`intro4/` と `lib/pico/`・`lib/mac/` を作り、各章の「6. コード」の Mac 側の解析ファイルを `%%writefile` で書き出します。中身は本文と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- **合成データ**：`synth_intro.py`（本文にはないファイル）が、本文の記録用紙・記録プログラムと同じ列の CSV（`led_iv.csv`、`data/bounce_S?.csv`、`joints.csv`、`data/noise_*.csv`、`data/env.csv`、`data/calib_S?.csv`）と、`sigrok-cli -O binary` と同じ形の `data/servo.bin` を作ります。真の値（GPIO の出力抵抗 40 Ω、Pico の温度の遅れ 120 秒、ヒステリシス −2° など）を決め、テスタの 1 mV 刻み・ADC の刻み・雑音を足したものです。真の値は `TRUTH` に入れ、ノートブックの確かめだけに使います（解析のプログラムには知らせません）。値は本文の「結果の例」の目安に合わせて選んだもので、実物の値ではありません。
- 各節では、ターミナルの `uv run python ○○.py …` の代わりに、`run("○○.py …", cwd="intro1")` で同じ引数のまま実行します。表示を見てから、`ok()` で本文の「確かめること」「結果の例」と照らし合わせます。
- **入れていないもの（実物が要る部分）**：Pico への書き込みと REPL（`blink.py`・`pwm_fade.py`・`button_toggle.py`・`bounce_logger.py`・`i2c_check.py`・`read_once.py`・`logger.py`・`servo_ctrl.py`）、シリアルで受け取る `bounce_capture.py`・`serial_logger.py`・`servo_tool.py`（`servo_tool.py current` の電流の図も、Pico が送る値をその場で描くので入れていません）、Raspberry Pi 5 での `sigrok-cli` の記録、はんだ付けと配線そのもの。自分の記録（CSV や `servo.bin`）を Colab の左のファイルの欄に上げて、同じ `run(...)` に名前を渡せば解析できるはずですが、執筆時には確かめていません。
- ⚠ 安全の注意（電池の短絡、はんだごて、サーボの電源など）は、教科書の各章の 2〜5 を読んでください。このノートブックは計算だけです。
- 乱数の種を変えなければ数値はほぼ同じですが、numpy・statsmodels の版で最後の桁が変わることがあります。Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels が入っている。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("statsmodels", "statsmodels"), ("scipy", "scipy"), ("pandas", "pandas")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import statsmodels, scipy, numpy, pandas
print("OK statsmodels", statsmodels.__version__, "scipy", scipy.__version__, "numpy", numpy.__version__, "pandas", pandas.__version__)
print("（ノートブックは statsmodels 0.15.0・scipy 1.18.1・numpy 2.5.5・pandas 3.1.1 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
import os
for d in ["intro1/data", "intro2", "intro3/data", "intro4/data", "lib/pico", "lib/mac"]:
    os.makedirs(d, exist_ok=True)
print("フォルダを作りました")

合成データを作る `synth_intro.py`（ノートブック版で追加。本文にはありません）を書き出します。

In [ ]:
%%writefile synth_intro.py
"""入門1〜4 のノートブック版：実物の測定の代わりに、本文のファイルと同じ形の合成データを作る（本文にはないファイル）

どの関数も「真の値」を決めて、そこにテスタ・ADC の刻みや雑音を足したものを書き出す。
真の値は TRUTH に入れ、ノートブックの確かめだけに使う（解析のプログラムには知らせない）。
"""
import csv
import pathlib

import numpy as np
from scipy.optimize import brentq

K_B = 1.380649e-23
Q_E = 1.602176634e-19
TRUTH = {}


# ---------------------------------------------------------------- 入門1：LED の電圧・電流
def led_rows(seed=1, temp_c=25.0):
    """GPIO（V_0 − R_out·I）→ 抵抗 → LED の回路を解いて、led_iv.csv の行を作る"""
    rng = np.random.default_rng(seed)
    vt = K_B * (temp_c + 273.15) / Q_E
    v0, r_out, r_burden = 3.300, 40.0, 10.0          # 真の値（R_out は「数十Ω」の目安の中から選んだ）
    leds = {"red": (2.0, 1.85), "green": (2.6, 2.55)}  # (理想係数 n, 4 mA での電圧 [V])
    TRUTH["led"] = dict(V0=v0, R_out=r_out, R_burden=r_burden, n={k: v[0] for k, v in leds.items()})
    rows = []
    for color, (n, v4) in leds.items():
        for r_nom in (330, 1000, 4700, 10000):
            r_true = r_nom * (1 + rng.normal(0, 0.004))           # 抵抗の個体差（±1% 品のつもり）

            def solve(extra):
                f = lambda v: v0 - 4e-3 * np.exp((v - v4) / (n * vt)) * (r_out + r_true + extra) - v
                v = brentq(f, 0.5, 3.3)
                return v, 4e-3 * np.exp((v - v4) / (n * vt))

            v_led, i = solve(0.0)
            meas = lambda v: round(v + rng.normal(0, 0.0015), 3)   # テスタ：雑音 1.5 mV、1 mV 刻み
            row = dict(color=color, R_nom=r_nom, R_meas_ohm=round(r_true * (1 + rng.normal(0, 0.001)), 1),
                       V_gpio=meas(v0 - i * r_out), V_R=meas(i * r_true), V_led=meas(v_led), I_meter_mA="")
            if color == "red" and r_nom in (330, 10000):          # 電流レンジ（負担 10 Ω のつもり）で測った値
                _, i_m = solve(r_burden)
                row["I_meter_mA"] = round(i_m * 1e3, 2 if i_m > 1e-3 else 3)
            rows.append(row)
    return rows


def append_csv(path, rows, cols):
    with open(path, "a", newline="") as f:
        f.write("\n")
        w = csv.DictWriter(f, fieldnames=cols)
        w.writerows(rows)


# ---------------------------------------------------------------- 入門1：チャタリング
def bounce(out_dir="data", presses=50, seed=2):
    """スイッチ3個 × 押す／離す。bounce_capture.py と同じ列の CSV を data/bounce_S?.csv に作る"""
    rng = np.random.default_rng(seed)
    # (チャタリングが起きる割合, 起きたときの継続時間の中央値 [µs])。スイッチの個体差と、押す／離すの差を入れる
    sw = {"S1": {"press": (0.70, 200), "release": (0.50, 120)},
          "S2": {"press": (0.65, 80), "release": (0.40, 80)},
          "S3": {"press": (0.85, 450), "release": (0.60, 250)}}
    TRUTH["bounce"] = sw
    pathlib.Path(out_dir).mkdir(exist_ok=True)
    for name, par in sw.items():
        rows, idx, t_ms = [], 0, 1500
        for i in range(presses):
            for kind in ("press", "release"):
                if i == presses - 1 and kind == "release":
                    break                    # bounce_capture.py と同じ：最後に押した時点で記録が終わり、最後の離すは入らない（99 行）
                p, med = par[kind]
                if rng.random() < p:
                    dur = int(min(3000, med * np.exp(rng.normal(0, 0.8))))   # 対数正規、3 ms で打ち切り
                    k = 1 + rng.poisson(dur / 100)                           # はね返りの回数
                    n_edges = 1 + 2 * k                                      # 1回の動作のエッジは奇数
                else:
                    dur, n_edges = 0, 1
                rows.append(dict(switch=name, idx=idx, kind=kind, n_edges=n_edges, dur_us=dur, t_ms=t_ms))
                idx += 1
                t_ms += int(rng.uniform(350, 650))
        with open(pathlib.Path(out_dir) / f"bounce_{name}.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["switch", "idx", "kind", "n_edges", "dur_us", "t_ms"])
            w.writeheader()
            w.writerows(rows)


# ---------------------------------------------------------------- 入門2：はんだ付けの検査
def joints(seed=3):
    """joints.csv の行。良品は真の値がみな同じ 0.10 Ω（ゼロ点を引いた値）で、当て直しで ±0.1 Ω 動く"""
    rng = np.random.default_rng(seed)
    z0 = 0.2
    bad = {"L07": 1.5, "L15": 3.1}          # 本文 入門2-3 のマスキングの数値例と同じ 1.5 Ω と 3.1 Ω
    cold = {"L04", "L12"}                   # 目視では ng だが導通はする（冷えたはんだのつもり）
    TRUTH["joints"] = dict(z0=z0, good=0.10, bad=bad, open="L20", cold=sorted(cold))
    step = lambda: rng.choice([-0.1, 0.0, 0.0, 0.1])     # 当て直しのばらつき（0.1 Ω 刻みの表示）
    rows = []
    for r in range(1, 6):
        rows.append(dict(item="zero", kind="zero", rep=r, ohm=f"{z0 + step():.1f}", visual=""))
    for i in range(1, 21):
        item = f"L{i:02d}"
        for r in range(1, 4):
            if item == "L20":
                ohm = "OL"
            else:
                ohm = f"{max(0.0, z0 + bad.get(item, 0.10) + step()):.1f}"
            vis = "ng" if (item in bad or item in cold or item == "L20") else "ok"
            rows.append(dict(item=item, kind="lead", rep=r, ohm=ohm, visual=vis if r == 1 else ""))
    for item, kind, v in [("P_red", "power", 0.15), ("P_black", "power", 0.15), ("PTC1", "ptc", 0.25)]:
        for r in range(1, 4):
            rows.append(dict(item=item, kind=kind, rep=r, ohm=f"{z0 + v + step():.1f}", visual="ok" if r == 1 else ""))
    return rows


# ---------------------------------------------------------------- 入門3：BME280・INA219 の記録
HEAD = ["host_time", "t_ms", "temp_c", "press_pa", "hum_pct", "pico_c", "bus_v", "shunt_mv", "current_ma"]


def _write(path, cols):
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    n = len(cols["t_ms"])
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(HEAD)
        for i in range(n):
            w.writerow([f"{cols['host_time'][i]:.3f}", int(cols["t_ms"][i]), f"{cols['temp_c'][i]:.2f}",
                        f"{cols['press_pa'][i]:.2f}", f"{cols['hum_pct'][i]:.3f}", f"{cols['pico_c'][i]:.2f}",
                        f"{cols['bus_v'][i]:.3f}", f"{cols['shunt_mv'][i]:.2f}", f"{cols['current_ma'][i]:.2f}"])


def _ar1(n, rho, sd, rng):
    e = rng.normal(0, sd, n)
    x = np.empty(n)
    x[0] = e[0] / np.sqrt(1 - rho ** 2)
    for i in range(1, n):
        x[i] = rho * x[i - 1] + e[i]
    return x


def _square_high(t, period, duty):
    """0 から t [s] までに、周期 period・デューティ duty の方形波が High だった時間の合計"""
    k = np.floor(t / period)
    return k * period * duty + np.minimum(t - k * period, period * duty)


def noise(path, seconds=120, period_ms=20, led="dc", ina_avg=1, seed=4):
    """logger.py の MODE = "noise" の記録（host_time つき）。led = "dc"（点けっぱなし）か "pwm"（1 kHz・50%）"""
    rng = np.random.default_rng(seed)
    n = int(seconds * 1000 / period_ms)
    t_ms = np.arange(n) * period_ms
    tt = t_ms / 1000
    temp = 26.30 + 0.03 * np.sin(2 * np.pi * tt / 300) + _ar1(n, 0.999, 0.0015, rng) + rng.normal(0, 0.003, n)
    press = 100800 + rng.normal(0, 1.5, n) + np.cumsum(rng.normal(0, 0.01, n))   # 白色雑音 1.5 Pa ＋ ゆっくりした揺らぎ
    i_on = 4.0                                     # LED に流れる電流 [mA]
    conv = {1: 0.532e-3, 128: 68.10e-3}[ina_avg]   # INA219 の変換時間（ina219.py の CONV_MS）
    if led == "dc":
        cur = i_on + rng.normal(0, 0.06, n)
    else:                                          # 変換の間の平均を読むとみなす（PWM とは同期していない）
        t0 = rng.uniform(0, 1, n) * 1e-3 + tt
        cur = i_on * (_square_high(t0 + conv, 1e-3, 0.5) - _square_high(t0, 1e-3, 0.5)) / conv + rng.normal(0, 0.03, n)
    shunt = np.round(cur * 0.1, 2)                 # 0.1 Ω、1目盛り 10 µV
    TRUTH[str(path)] = dict(white_press_pa=1.5, period_ms=period_ms, led=led, ina_avg=ina_avg)
    _write(path, dict(host_time=1.7e9 + tt + rng.uniform(0, 0.004, n), t_ms=t_ms, temp_c=np.round(temp, 2),
                      press_pa=np.round(press, 2), hum_pct=45 + rng.normal(0, 0.02, n), pico_c=np.zeros(n),
                      bus_v=np.round((3.152 + rng.normal(0, 0.002, n)) / 0.004) * 0.004,
                      shunt_mv=shunt, current_ma=np.round(shunt / 0.1, 2)))


def env(path, minutes=120, lag_s=120.0, offset=3.0, seed=5):
    """logger.py の MODE = "env"（1秒ごと）。Pico の内蔵センサは時定数 lag_s の1次遅れ＋offset＋雑音 0.7 °C"""
    rng = np.random.default_rng(seed)
    n = int(minutes * 60)
    tt = np.arange(n, dtype=float)
    room = 23.0 + 2.5 * tt / tt[-1] + 0.8 * np.sin(2 * np.pi * tt / 1200) + _ar1(n, 0.995, 0.005, rng)
    pico_true = np.empty(n)
    pico_true[0] = room[0]
    a = 1.0 / lag_s
    for i in range(1, n):
        pico_true[i] = pico_true[i - 1] + a * (room[i] - pico_true[i - 1])
    TRUTH["env"] = dict(lag_s=lag_s, offset=offset, slope=1.0)
    _write(path, dict(host_time=1.7e9 + tt + rng.uniform(0, 0.01, n), t_ms=tt * 1000,
                      temp_c=np.round(room + rng.normal(0, 0.01, n), 2),
                      press_pa=np.round(100800 + rng.normal(0, 1.5, n), 2), hum_pct=45 + rng.normal(0, 0.2, n),
                      pico_c=np.round(pico_true + offset + rng.normal(0, 0.7, n), 2),
                      bus_v=np.full(n, 3.152), shunt_mv=np.full(n, 0.40), current_ma=np.full(n, 4.0)))


# ---------------------------------------------------------------- 入門4：ロジックアナライザとサーボ
def servo_bin(path, fs=1_000_000, seconds=4.0, seed=6):
    """sigrok-cli -O binary と同じ形（1サンプル1バイト、ビット0 = D0）。servo_tool.py seq の 1000・1500・2000 µs"""
    rng = np.random.default_rng(seed)
    n = int(fs * seconds)
    tick = 46 / 150e6                    # Pico 2 W（150 MHz）で 50 Hz を出すときの分周（46）の1カウント [s]
    period = 0.020 * (1 + 30e-6)         # Pico とロジアナの時計の差を +30 ppm とした
    raw = ((np.arange(n) // 37) % 2 << 1).astype(np.uint8)   # ビット1 = D1 に別の信号（D0 だけを取り出せるか）
    t = rng.uniform(0, period)
    seq_t0 = rng.uniform(0, 3.0)         # seq の途中から記録を始める
    widths = []
    while t < seconds:
        cmd = (1000, 1500, 2000)[int((seq_t0 + t) // 1.0) % 3]
        w = np.round(cmd * 1e-6 / tick) * tick
        a, b = int(np.ceil(t * fs)), int(np.ceil((t + w) * fs))   # 次のサンプルで初めて 1（0）になる
        if b < n:
            raw[a:b] |= 1
            widths.append((cmd, w * 1e6))
        t += period
    raw.tofile(path)
    TRUTH["servo_bin"] = dict(period_ms=period * 1e3, tick_us=tick * 1e6)


def calib(path, servo, b, a90, h, seed):
    """servo_tool.py calib と同じ列。角度 = a90 + b·(x−1500) + h·(向き ±1/2)、読み取り 0.4°、0.5° 刻み"""
    rng = np.random.default_rng(seed)
    pulses = list(range(1000, 2001, 100))
    rows = []
    for rep in (1, 2, 3):
        for direction, seq in (("up", pulses[1:]), ("down", pulses[::-1][1:])):   # 端は手前から近づけないのでとばす
            for us in seq:
                ang = a90 + b * (us - 1500) + h * (0.5 if direction == "up" else -0.5) + rng.normal(0, 0.4)
                rows.append([servo, rep, direction, us, round(ang * 2) / 2])
    TRUTH[f"calib_{servo}"] = dict(b=b, a90=a90, h=h)
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["servo", "rep", "direction", "pulse_us", "angle_deg"])
        w.writerows(rows)

In [ ]:
import glob, json, os, re, shlex, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import Image, display

HOME = os.getcwd()   # このノートブックのフォルダ（教科書の ~/home-lab の代わり）


def run(args, cwd="."):
    """ターミナルの「uv run python ○○.py …」の代わり。cwd のフォルダで実行し、表示を文字列でも返す"""
    wd = os.path.join(HOME, cwd)
    words = []
    for w in shlex.split(args):   # data/bounce_*.csv のような * を、ターミナルと同じく名前の順に広げる
        hits = sorted(glob.glob(os.path.join(wd, w))) if any(c in w for c in "*?[") else []
        words += [os.path.relpath(p, wd) for p in hits] or [w]
    p = subprocess.run([sys.executable] + words, capture_output=True, text=True, cwd=wd)
    print(p.stdout, end="")
    keep = [l for l in p.stderr.splitlines() if "findfont" not in l and l.strip()]   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）
    if keep:
        print("\n".join(keep), file=sys.stderr)
    return p.stdout, p.returncode


def show(*paths, width=800):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=os.path.join(HOME, f), width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i).replace("−", "-").replace("+", ""))


print("準備ができました")


import synth_intro as syn

## 1. 入門1 の数値例（入門1-3）

本文の数値例を、そのまま計算して確かめます。LED の抵抗は $R = (V_{OH}-V_F)/I$、電流計の負担電圧（内部の抵抗を仮に 10 Ω）、PWM の平均電圧 $\bar v = DV_{OH}$、明るさの感じ方 $L^* = 116Y^{1/3}-16$、ガンマ補正 $D_k = (k/K)^{\gamma}$ です。

In [ ]:
VOH, VF = 3.3, 1.9
R = (VOH - VF) / 0.004
I330 = (VOH - VF) / 330
P330 = I330 ** 2 * 330
I_burden = (VOH - VF) / (330 + 10)
print(f"R = {R:.0f} Ω、330 Ω で I = {I330*1e3:.2f} mA、P = {P330*1e3:.2f} mW、テスタ（10 Ω）を入れると {I_burden*1e3:.2f} mA（{(1 - I_burden/I330)*100:.1f}% 小さい）")
Lstar = 116 * 0.5 ** (1 / 3) - 16
D1 = (1 / 7) ** 2.2
print(f"D = 0.5 の L* = {Lstar:.1f}、K = 7 の1段目 D1 = {D1:.4f} → duty_u16 = {D1*65535:.0f}、D = 50% の平均電圧 {0.5*VOH:.2f} V")
ok(abs(R - 350) < 0.5, "R = 1.4/0.004 = 350 Ω")
ok(abs(I330 * 1e3 - 4.2) < 0.05 and abs(P330 * 1e3 - 5.9) < 0.05, "330 Ω で約 4.2 mA、抵抗の電力は約 5.9 mW（1/4 W より十分小さい）")
ok(abs(I_burden * 1e3 - 4.1) < 0.05 and abs((1 - I_burden / I330) * 100 - 3) < 0.5, "負担 10 Ω で約 4.1 mA、約3% 小さい")
ok(abs(Lstar - 76) < 0.5, "D = 0.5 でも L* ≈ 76")
ok(abs(D1 - 0.0138) < 0.0001 and abs(D1 * 65535 - 906) < 1.5, "1段目は (1/7)^2.2 ≈ 0.0138、duty_u16 で約 906")
ok(abs(0.5 * VOH - 1.65) < 1e-9, "D = 50% でテスタの直流電圧は約 1.65 V")

## 2. LED の電圧・電流の解析（入門1 手順13〜16）

本文の記録用紙 `led_iv.csv`（見出しだけ）を書き出し、合成の測定値を足します。合成では、GPIO を「$V_0 = 3.300$ V の電源と出力抵抗 $R_{\mathrm{out}} = 40$ Ω」、LED を理想のダイオードの式（赤 $n = 2.0$、緑 $n = 2.6$）とし、抵抗 330 Ω・1 kΩ・4.7 kΩ・10 kΩ（手順12・16 と同じ4本）の回路を解いて、3つの電圧をテスタの 1 mV 刻み（雑音 1.5 mV）で読んだことにしました。電流レンジの値（`I_meter_mA`）は、赤の 330 Ω と 10 kΩ だけ、負担 10 Ω を入れて解いた値です。

In [ ]:
%%writefile intro1/led_iv.csv
# 入門1 LED の電圧・電流の記録用紙（# で始まる行は読み飛ばされる）
# color      : LED の色（red, green など半角英字）
# R_nom      : 抵抗の公称値 [Ω]（カラーコードの値）
# R_meas_ohm : 回路に挿す前にテスタで測った抵抗値 [Ω]
# V_gpio     : GP15（ピン20）〜 GND の電圧 [V]
# V_R        : 抵抗の両端の電圧 [V]
# V_led      : LED の両端の電圧（アノード〜カソード）[V]
# I_meter_mA : テスタの電流レンジで測った電流 [mA]（測らないときは空欄のまま）
# 下の見出し行は消さない。1回測るごとに1行足す（値はすべて自分の測定値を書く）
color,R_nom,R_meas_ohm,V_gpio,V_R,V_led,I_meter_mA

In [ ]:
syn.append_csv("intro1/led_iv.csv", syn.led_rows(), ["color", "R_nom", "R_meas_ohm", "V_gpio", "V_R", "V_led", "I_meter_mA"])
print(open("intro1/led_iv.csv", encoding="utf-8").read())

In [ ]:
%%writefile intro1/led_iv.py
#!/usr/bin/env python3
"""入門1：LED の電圧・電流の測定値（led_iv.csv）を解析する

使い方: cd ~/home-lab/intro1 && uv run python led_iv.py led_iv.csv --temp 25
  --temp は測ったときの室温 [°C]（熱電圧 V_T の計算に使う）

やること
  1. 抵抗の両端電圧 V_R と抵抗の実測値から電流 I = V_R / R を計算する
  2. キルヒホッフの電圧則 V_gpio = V_R + V_led がどれだけ成り立つかを確かめる
  3. 色ごとに ln I = a + b V_led を最小二乗で当てはめ、理想係数 n = 1/(b V_T) の目安を出す
  4. GPIO の出力電圧 V_gpio と電流 I の直線から、GPIO の出力抵抗を推定する
  5. （I_meter_mA の列があれば）テスタの電流レンジで測った値と比べる
"""
import argparse
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

K_B = 1.380649e-23      # ボルツマン定数 [J/K]
Q_E = 1.602176634e-19   # 電気素量 [C]
NEED = ["color", "R_nom", "R_meas_ohm", "V_gpio", "V_R", "V_led"]


def load(path):
    df = pd.read_csv(path, comment="#", skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    miss = [c for c in NEED if c not in df.columns]
    if miss:
        sys.exit(f"列が足りません: {miss}（1行目の見出しを led_iv.csv の見本どおりにしてください）")
    df = df.dropna(subset=NEED).copy()
    if df.empty:
        sys.exit("データの行がありません。測った値を led_iv.csv に書き込んでください")
    df["color"] = df["color"].astype(str).str.strip()
    df["I_mA"] = df["V_R"] / df["R_meas_ohm"] * 1e3
    # 0.001 mV で丸める（浮動小数点の誤差で 0 が -2.22e-13 と表示されないように。+ 0.0 は -0 を 0 にする）
    df["kvl_mV"] = ((df["V_gpio"] - df["V_R"] - df["V_led"]) * 1e3).round(3) + 0.0
    return df


def fit_led(g, vt):
    """ln I = a + b V を当てはめ、n = 1/(b V_T) とその95%区間を返す"""
    x = g["V_led"].to_numpy()
    y = np.log(g["I_mA"].to_numpy())
    fit = sm.OLS(y, sm.add_constant(x)).fit()
    b = fit.params[1]
    lo, hi = fit.conf_int(alpha=0.05)[1]
    n_hat = 1.0 / (b * vt)
    # n は b の単調減少関数なので、b の区間の端を入れ替えて変換すればよい
    n_lo = 1.0 / (hi * vt) if hi > 0 else np.nan
    n_hi = 1.0 / (lo * vt) if lo > 0 else np.inf
    return fit, n_hat, n_lo, n_hi


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("--temp", type=float, default=25.0, help="室温 [°C]")
    a = ap.parse_args()

    df = load(a.csv)
    vt = K_B * (a.temp + 273.15) / Q_E
    print(f"室温 {a.temp:.1f} °C → 熱電圧 V_T = {vt*1e3:.2f} mV\n")

    show = df[["color", "R_nom", "R_meas_ohm", "V_gpio", "V_R", "V_led", "I_mA", "kvl_mV"]]
    print(show.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    worst = df["kvl_mV"].abs().max()
    print(f"\nキルヒホッフの電圧則のずれ |V_gpio − V_R − V_led| の最大 = {worst:.1f} mV", end="")
    # 3つの電圧は別々に測るので、テスタの分解能と確度（DE-200A の取扱説明書で要確認）の分だけずれる
    print("（20mV 未満なら、テスタの誤差の範囲で整合している）" if worst < 20
          else "（大きい。測る点の取り違えや読み間違いを確認）")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))

    # ---- 1. LED の I-V（片対数） ----
    print("\n[LED] ln I = a + b·V_led の当てはめ（点が3つ以上ある色だけ）")
    for color, g in df.groupby("color"):
        g = g.sort_values("V_led")
        axes[0].semilogy(g["V_led"], g["I_mA"], "o-", label=color)
        if len(g) < 3:
            print(f"  {color}: 点が {len(g)} 個なので当てはめは省略")
            continue
        fit, n_hat, n_lo, n_hi = fit_led(g, vt)
        dec_mv = np.log(10) / fit.params[1] * 1e3
        print(f"  {color}: 電流が10倍になるのに必要な電圧 = {dec_mv:.0f} mV、"
              f"n ≈ {n_hat:.2f}（95%区間 {n_lo:.2f}〜{n_hi:.2f}、点 {len(g)} 個）")
        if not np.isfinite(n_hi):
            print("         傾きの区間が0をまたぐので、n の上限は決まらない（点が少なすぎる）")
    axes[0].set_xlabel("LED の電圧 V_led [V]")
    axes[0].set_ylabel("電流 I [mA]（対数目盛）")
    axes[0].set_title("LED の電圧と電流")
    axes[0].grid(True, which="both", alpha=0.3)
    axes[0].legend()

    # ---- 2. GPIO の出力抵抗 ----
    x = df["I_mA"].to_numpy()
    y = df["V_gpio"].to_numpy()
    print("\n[GPIO] V_gpio = V_0 − R_out·I の当てはめ")
    if len(df) >= 3 and np.ptp(x) > 0:
        fit = sm.OLS(y, sm.add_constant(x)).fit()
        v0, slope = fit.params
        (v0_lo, v0_hi), (s_lo, s_hi) = fit.conf_int(alpha=0.05)
        print(f"  電流0のときの出力電圧 V_0 = {v0:.4f} V（95%区間 {v0_lo:.4f}〜{v0_hi:.4f}）")
        print(f"  出力抵抗 R_out = {-slope*1e3:.1f} Ω（95%区間 {-s_hi*1e3:.1f}〜{-s_lo*1e3:.1f}）")
        xx = np.linspace(0, x.max() * 1.05, 50)
        axes[1].plot(xx, v0 + slope * xx, "-", color="gray", label=f"直線 R_out≈{-slope*1e3:.0f} Ω")
    else:
        print("  点が足りないので省略")
    for color, g in df.groupby("color"):
        axes[1].plot(g["I_mA"], g["V_gpio"], "o", label=color)
    axes[1].set_xlabel("電流 I [mA]")
    axes[1].set_ylabel("GP15 の電圧 V_gpio [V]")
    axes[1].set_title("GPIO の出力電圧は電流で下がる")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    # ---- 3. 電流計との比較（任意） ----
    if "I_meter_mA" in df.columns and df["I_meter_mA"].notna().any():
        m = df.dropna(subset=["I_meter_mA"])
        ratio = m["I_meter_mA"] / m["I_mA"]
        print("\n[電流計] テスタの電流レンジの値 ÷ 抵抗の電圧から求めた値")
        for (_, r), q in zip(m.iterrows(), ratio):
            print(f"  {r['color']:6s} R={r['R_nom']:>6}: {r['I_meter_mA']:.4g} mA / {r['I_mA']:.4g} mA = {q:.3f}")
        print("  1より小さいのは、電流計の内部の抵抗（負担電圧）で電流が減ったため。抵抗が小さいほど影響が大きい")

    fig.tight_layout()
    fig.savefig("led_iv.png", dpi=150)
    print("\nled_iv.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
out_led, rc = run("led_iv.py led_iv.csv --temp 25", cwd="intro1")
show("intro1/led_iv.png")

In [ ]:
# 確かめ（手順14・16 と 入門1-7 の結果の例）
ok(rc == 0, "led_iv.py がエラーなく終わった")
ok(abs(num(r"V_T = ([\d.]+) mV", out_led) - 25.69) < 0.01, "25 °C の熱電圧 V_T = 25.69 mV")
d = pd.read_csv("intro1/led_iv.csv", comment="#", skipinitialspace=True)
d["I_mA"] = d["V_R"] / d["R_meas_ohm"] * 1e3
red = d[d["color"] == "red"].set_index("R_nom")
ok(3.5 < red.loc[330, "I_mA"] < 4.6, f"赤・330 Ω で約 4 mA（{red.loc[330, 'I_mA']:.2f} mA）")
ok(0.12 < red.loc[10000, "I_mA"] < 0.2, f"赤・10 kΩ で約 0.15 mA（{red.loc[10000, 'I_mA']:.3f} mA）")
dv = red.loc[330, "V_led"] - red.loc[10000, "V_led"]
ok(0.1 < dv < 0.3, f"電流が約 {red.loc[330, 'I_mA']/red.loc[10000, 'I_mA']:.0f} 倍に変わっても、V_led の変化は {dv:.2f} V（0.2 V 程度）")
ok(num(r"の最大 = ([\d.]+) mV", out_led) < 20, "キルヒホッフの電圧則のずれの最大が 20 mV 未満")
Rout = num(r"出力抵抗 R_out = ([\d.]+) Ω", out_led)
lo, hi = num(r"R_out = [\d.]+ Ω（95%区間 ([\d.]+)〜", out_led), num(r"R_out = [\d.]+ Ω（95%区間 [\d.]+〜([\d.]+)）", out_led)
ok(10 < Rout < 100 and lo - 0.5 < syn.TRUTH["led"]["R_out"] < hi + 0.5,
   f"R_out = {Rout} Ω（数十Ω）で、95%区間 {lo}〜{hi} Ω が真の値 40 Ω を含む（区間の端の丸めの分 0.5 Ω の余裕）")
n_red = num(r"red: .*?n ≈ ([\d.]+)", out_led)
ok(abs(n_red - 2.0) < 0.2, f"赤の n ≈ {n_red}（合成の真の値 2.0）")
r330, r10k = num(r"R=   330: .*?= ([\d.]+)", out_led), num(r"R= 10000: .*?= ([\d.]+)", out_led)
ok(0.95 < r330 < 0.99 and r10k > 0.99, f"電流計の値の比：330 Ω で {r330}（数% 小さい）、10 kΩ で {r10k}（ほとんど差がない）")

**合成データでは $n$ の区間が狭く出ます。** 合成の LED は理想のダイオードの式どおりで、4点がほぼ一直線に並ぶからです。本文の 入門1-7 の「4点だけから求めた $n$ は、区間がとても広くなるか、上限が決まらないことがある」は、実物の LED（直列抵抗や温度で式からずれる）とテスタの誤差がある場合の話です。

## 3. チャタリングの継続時間（入門1 段階5、入門1-3-7）

`bounce_capture.py` が保存する `data/bounce_S?.csv`（列 `switch,idx,kind,n_edges,dur_us,t_ms`）と同じ形のものを、スイッチ3個 × 50回押す分だけ合成します（本文の手順と同じく、50回目を押した時点で記録が終わるので、離すは 49 回、1個あたり 99 行です）。スイッチごとに「チャタリングが起きる割合」と「起きたときの継続時間（対数正規分布）」を変え、押すときと離すときでも変えてあります。はね返りの回数 $k$ からエッジの数を $1 + 2k$（奇数）にしています。

In [ ]:
syn.bounce(out_dir="intro1/data", presses=50)
for k, v in syn.TRUTH["bounce"].items():
    print(k, v, "（チャタリングが起きる割合, 起きたときの継続時間の中央値 µs）")

In [ ]:
%%writefile intro1/bounce_analysis.py
#!/usr/bin/env python3
"""入門1：チャタリングの継続時間を解析する

使い方: cd ~/home-lab/intro1 && uv run python bounce_analysis.py data/bounce_*.csv

やること
  1. スイッチごと・押す／離すごとに、チャタリングが起きた割合（Wilson区間）と継続時間の分布をまとめる
  2. 分位点（中央値・95%点・99%点）の、分布の形を仮定しない信頼区間を順序統計量から求める
  3. 「全部の押下をひとまとめにした区間」と「スイッチ単位で再標本化した区間」を比べる（疑似反復）
  4. スイッチの間の違い（クラスカル・ウォリス検定）、押す／離すの違い（マン・ホイットニー検定＋Holm補正）
  5. ソフトウェアのデバウンス時間の目安を出す
"""
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

RNG = np.random.default_rng(1)
N_BOOT = 4000


def quantile_ci(x, p, conf=0.95):
    """p 分位点の信頼区間（分布の形を仮定しない。順序統計量と二項分布を使う）

    x を小さい順に並べた x_(1) ≤ … ≤ x_(n) について、
    P(x_(l) ≤ 真の p 分位点 < x_(u)) = P(l ≤ B ≤ u−1)、B ~ 二項分布(n, p)。
    端が決められないとき（データが足りないとき）は None を返す。
    """
    x = np.sort(np.asarray(x))
    n = len(x)
    alpha = 1.0 - conf
    l = int(stats.binom.ppf(alpha / 2, n, p))          # P(B ≤ l−1) < α/2 となる l
    u = int(stats.binom.ppf(1 - alpha / 2, n, p)) + 1  # P(B ≥ u) ≤ α/2 となる u
    lo = x[l - 1] if l >= 1 else None
    hi = x[u - 1] if u <= n else None
    return lo, hi


def fmt(v):
    return "決まらない" if v is None else f"{v/1000:.3f}"


def ecdf(ax, x, label):
    x = np.sort(np.asarray(x) / 1000.0)
    y = np.arange(1, len(x) + 1) / len(x)
    ax.step(x, y, where="post", label=label)


def main(paths):
    if not paths:
        sys.exit("CSV を指定してください（例: data/bounce_*.csv）")
    df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    df["chatter"] = df["n_edges"] > 1
    df["odd_ok"] = df["n_edges"] % 2 == 1          # 1回の動作のエッジ数は本来奇数
    switches = sorted(df["switch"].unique())
    print(f"読み込み: {len(paths)} ファイル、{len(df)} 件、スイッチ {switches}\n")

    # ---------- 1・2. スイッチ × 押す／離す ごとのまとめ ----------
    print("単位: 継続時間は ms。区間はすべて95%")
    print(f"{'スイッチ':6s} {'動作':7s} {'件数':>4s} {'チャタあり[%]':>12s} {'(Wilson区間)':>14s} "
          f"{'中央値':>7s} {'95%点':>7s} {'最大':>7s} {'99%点の上限':>10s} {'偶数エッジ':>8s}")
    for (sw, kind), g in df.groupby(["switch", "kind"]):
        n = len(g)
        k = int(g["chatter"].sum())
        lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
        d = g["dur_us"].to_numpy()
        q99_lo, q99_hi = quantile_ci(d, 0.99)
        print(f"{sw:6s} {kind:7s} {n:4d} {k/n*100:12.1f} {f'({lo*100:.0f}〜{hi*100:.0f})':>14s} "
              f"{np.median(d)/1000:7.3f} {np.quantile(d, 0.95)/1000:7.3f} {d.max()/1000:7.3f} "
              f"{fmt(q99_hi):>10s} {int((~g['odd_ok']).sum()):8d}")
    print("\n・「99%点の上限」が「決まらない」のは、件数が少なくて最大値すら99%点を超えている保証がないため。")
    print("   最大値が95%の確からしさで99%点以上になるには、1 − 0.99^n ≥ 0.95 より n ≥ 299 回が必要（片側）。")
    print("   この表の上限は両側95%区間の上の端なので、0.99^n ≤ 0.025 より n ≥ 368 回で決まる。")
    n_even = int((~df["odd_ok"]).sum())
    if n_even:
        print(f"・エッジ数が偶数の件が {n_even} 件ある。速いエッジを取りこぼしている（ロジックアナライザで確かめる）。")

    press = df[df["kind"] == "press"]
    lo, hi = quantile_ci(press["dur_us"], 0.5)
    print(f"\n押したとき（全スイッチ）の中央値 = {np.median(press['dur_us'])/1000:.3f} ms"
          f"（順序統計量による95%区間 {fmt(lo)}〜{fmt(hi)} ms）")

    # ---------- 3. 疑似反復：ひとまとめ vs スイッチ単位 ----------
    groups = [g["dur_us"].to_numpy() for _, g in press.groupby("switch")]
    allx = np.concatenate(groups)
    naive = [np.median(RNG.choice(allx, allx.size)) for _ in range(N_BOOT)]
    print("\n[疑似反復] 「このタクトスイッチという部品」の中央値を知りたいときの区間")
    print(f"  誤り：全押下を独立とみなしたブートストラップ  "
          f"{np.percentile(naive, 2.5)/1000:.3f}〜{np.percentile(naive, 97.5)/1000:.3f} ms（n = {allx.size} 回）")
    if len(groups) >= 2:
        clus = []
        for _ in range(N_BOOT):
            pick = RNG.integers(0, len(groups), len(groups))           # スイッチを選び直す
            res = np.concatenate([RNG.choice(groups[i], groups[i].size) for i in pick])
            clus.append(np.median(res))
        print(f"  正しい：スイッチ単位で選び直すブートストラップ "
              f"{np.percentile(clus, 2.5)/1000:.3f}〜{np.percentile(clus, 97.5)/1000:.3f} ms（独立な単位は {len(groups)} 個）")
        print("  後者のほうが広いのが普通。スイッチが少ない（5個以下）と、この区間自体も粗いことに注意。")
    else:
        print("  スイッチが1個だけなので、部品としての区間は求められない（1個の性質しか分からない）。")

    # ---------- 4. 検定 ----------
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        print(f"\n[スイッチ間の違い] クラスカル・ウォリス検定（押したとき）: H = {h:.2f}, p = {p:.3g}")
        print("  p が小さければ「スイッチの個体によって継続時間の分布が違う」と言える。")
    pv, names = [], []
    for sw, g in df.groupby("switch"):
        a = g.loc[g["kind"] == "press", "dur_us"]
        b = g.loc[g["kind"] == "release", "dur_us"]
        if len(a) >= 5 and len(b) >= 5:
            pv.append(stats.mannwhitneyu(a, b, alternative="two-sided").pvalue)
            names.append(sw)
    if pv:
        rej, p_holm, _, _ = multipletests(pv, alpha=0.05, method="holm")
        print("\n[押す vs 離す] マン・ホイットニーのU検定（スイッチごと。Holm法で多重比較を補正）")
        for sw, p0, p1, r in zip(names, pv, p_holm, rej):
            print(f"  {sw}: p = {p0:.3g} → 補正後 {p1:.3g}  {'差あり' if r else '差があるとは言えない'}")

    # ---------- 5. デバウンス時間の目安 ----------
    mx = df["dur_us"].max() / 1000
    rec = max(5, int(np.ceil(mx * 2)))
    _, q99_all = quantile_ci(df["dur_us"], 0.99)
    print(f"\n[デバウンス時間の目安] 観測した最大の継続時間 {mx:.2f} ms の2倍を切り上げ、5 ms 未満なら 5 ms → {rec} ms")
    print("  「2倍」は余裕を見るための経験則（安全係数）で、保証ではない（最大値は99%点の保証にならないことは上で見たとおり）。")
    if q99_all is None:
        print(f"  参考：全件（{len(df)} 件）を独立とみなしても、99%点の95%上限は件数が足りず決まらない（両側95%区間の上の端には 368 件以上が必要）。")
    else:
        print(f"  参考：全件（{len(df)} 件）の99%点の95%上限 = {q99_all/1000:.3f} ms（目安がこれより長いことを確かめる）。")
        print("  この上限は全件を独立とみなした参考値で、スイッチの個体差（上の疑似反復）を無視しているぶん狭めに出る。")
    print("  ただし、これは手元のスイッチと押し方での値。古くなった接点や別の部品では長くなることがある。")

    # ---------- 図 ----------
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    for ax, kind, title in [(axes[0], "press", "押したとき"), (axes[1], "release", "離したとき")]:
        for sw, g in df[df["kind"] == kind].groupby("switch"):
            ecdf(ax, g["dur_us"], sw)
        ax.set_xscale("symlog", linthresh=0.01)
        ax.set_xlabel("チャタリングの継続時間 [ms]（0.01 ms 以下は直線目盛）")
        ax.set_title(title)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend()
    axes[0].set_ylabel("累積割合（経験分布関数）")
    fig.tight_layout()
    fig.savefig("bounce.png", dpi=150)
    print("\nbounce.png を保存しました")


if __name__ == "__main__":
    main(sys.argv[1:])

In [ ]:
out_b, rc = run("bounce_analysis.py data/bounce_*.csv", cwd="intro1")
show("intro1/bounce.png")

In [ ]:
# 確かめ（入門1-7 の段階5 の結果の例）
ok(rc == 0, "bounce_analysis.py がエラーなく終わった")
tab = [l for l in out_b.splitlines() if re.match(r"S\d +(press|release)", l)]
ok(len(tab) == 6 and all("決まらない" in l for l in tab), "スイッチ・動作ごとに 50 回（離すは 49 回）では「99%点の上限」は「決まらない」")
ok("偶数の件" not in out_b, "エッジ数が偶数の件はない（合成では取りこぼしを入れていない）")
nai = [num(r"独立とみなしたブートストラップ +([\d.]+)〜", out_b), num(r"独立とみなしたブートストラップ +[\d.]+〜([\d.]+) ms", out_b)]
clu = [num(r"スイッチ単位で選び直すブートストラップ ([\d.]+)〜", out_b), num(r"スイッチ単位で選び直すブートストラップ [\d.]+〜([\d.]+) ms", out_b)]
ok(clu[1] - clu[0] > 2 * (nai[1] - nai[0]), f"疑似反復：スイッチ単位の区間 {clu} ms のほうが、ひとまとめの区間 {nai} ms より広い")
ok(num(r"H = [\d.]+, p = ([\d.e+-]+)", out_b) < 0.01, "スイッチの間の違いはクラスカル・ウォリス検定で有意（合成では中央値を 80〜450 µs と変えた）")
ok(re.search(r"→ (\d+) ms", out_b) and int(re.search(r"→ (\d+) ms", out_b).group(1)) >= 5, "デバウンス時間の目安は 5 ms 以上")

**「99%点の上限」が決まるのに必要な件数について。** 本文の 入門1-3-7 (2) は、最大値が 95% の確からしさで 99%点以上になる条件 $1-0.99^n \ge 0.95$ から「299回以上」とし、続けて、`quantile_ci` は**両側** 95% の区間（上側に 2.5% ずつ）なので、上の端が決まるのは $0.99^n \le 0.025$、つまり $n \ge 368$ のときだと説明しています。上の表示でも、3個を合わせた全件（297 件）では「決まらない」と出ています。次のセルで確かめます。

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("bounce_analysis", "intro1/bounce_analysis.py")
ba = importlib.util.module_from_spec(spec); spec.loader.exec_module(ba)
x = np.arange(1, 400, dtype=float)
n_need = next(n for n in range(2, 400) if ba.quantile_ci(x[:n], 0.99)[1] is not None)
print(f"片側 95%：ln 0.05 / ln 0.99 = {np.log(0.05)/np.log(0.99):.1f} → 299 回（本文）")
print(f"両側 95%（quantile_ci の上の端）：ln 0.025 / ln 0.99 = {np.log(0.025)/np.log(0.99):.1f} → {n_need} 回")
ok(n_need == 368, "quantile_ci で 99%点の上限が決まるのは 368 件から（299 件は片側の場合）")
ok(ba.quantile_ci(x[:300], 0.99)[1] is None, "300 件では、まだ「決まらない」")

## 4. はんだ付けの検査（入門2 手順25、入門2-3）

まず本文 入門2-3 の数値例（電線の電圧降下、短絡電流とヒューズ、マスキング）を計算します。

In [ ]:
drop = 0.055 * 0.6 * 1
Isc = 5.6 / 0.17
x = np.array([0.1] * 10 + [0.2] * 5 + [0.0] * 2 + [1.5, 3.1])
mu, sd = x.mean(), x.std(ddof=1)
print(f"電圧降下 {drop:.3f} V、短絡電流 {Isc:.0f} A、マスキングの例：平均 {mu:.3f} Ω、標準偏差 {sd:.3f} Ω、"
      f"3.1 Ω の z = {(3.1-mu)/sd:.2f}、1.5 Ω の z = {(1.5-mu)/sd:.2f}")
ok(abs(drop - 0.03) < 0.005, "AWG22 往復 60 cm に 1 A で約 0.03 V")
ok(abs(Isc - 33) < 0.5, "満充電 5.6 V・0.17 Ω の短絡電流は約 33 A")
# ヒューズ FU1（2A、冷抵抗 約0.07 Ω・溶断 I²t 約5.2 A²s は Littelfuse 312 の公称値を目安に）
I_xt = 5.6 / (0.17 + 0.07)
I_p2 = 5.6 / (0.12 + 0.07 + 0.11 + 0.1)
R4 = 0.12 + 0.07 + 0.15 + 0.1 + 0.15
i2t_in = 5.6**2 * 470e-6 / (2 * R4)
print(f"FU1 あり：XT30・P1 の短絡 {I_xt:.0f} A（約 {5.2/I_xt**2*1e3:.0f} ms）、P2〜P4 の短絡 {I_p2:.0f} A（約 {5.2/I_p2**2*1e3:.0f} ms）、"
      f"470 µF の突入 I²t = {i2t_in:.4f} A²s（溶断の 1/{5.2/i2t_in:.0f}）、AWG22 の k²S² = {115**2*0.33**2:.0f} A²s")
ok(abs(I_xt - 23) < 0.5 and abs(I_p2 - 14) < 0.5, "ヒューズの 0.07 Ω を足すと短絡電流は約 23 A・約 14 A（どちらも 2A の 7 倍以上）")
ok(abs(i2t_in - 0.013) < 0.001 and 300 < 5.2 / i2t_in < 500, "470 µF の突入の I²t ≈ 0.013 A²s（溶けるのに要る値の約 1/400）")
ok(abs(mu - 0.35) < 0.01 and abs(sd - 0.74) < 0.01, "19本の例：平均 約 0.35 Ω、標準偏差 約 0.74 Ω")
ok(abs((3.1 - mu) / sd - 3.7) < 0.05 and abs((1.5 - mu) / sd - 1.6) < 0.05, "3.1 Ω は z ≈ 3.7（外れ値）、1.5 Ω は z ≈ 1.6（見逃す）")

次に、記録用紙 `joints.csv` に合成の測定値を足します。ゼロ点（テスト棒どうし）5回、練習用リード L01〜L20 を3回ずつ、電源リード2本とポリスイッチ1個です。良品のリードは**真の値がみな同じ**（ゼロ点を引いて 0.10 Ω）で、当て直しで表示が ±0.1 Ω 動くだけにしました。L07 と L15 には、上のマスキングの例と同じ 1.5 Ω と 3.1 Ω を足し、L20 は開放（`OL`）、L04・L12 は「目視では ng だが導通はする」ものです。

In [ ]:
%%writefile intro2/joints.csv
# 入門2 はんだ付けの検査の記録（# で始まる行は読み飛ばされる）
# item   : 測ったものの名前（zero = テスト棒どうしを当てたとき、L01〜L20 = 練習用リード、など）
# kind   : zero（テスト棒どうし）/ lead（練習用リード）/ power（電源リード）/ ptc（ポリスイッチ）
# rep    : 何回目か（1, 2, 3）。1回ごとにテスト棒を離して当て直す
# ohm    : テスタの表示 [Ω]。範囲外（OL など）と表示されたら OL と書く
# visual : 目視の判定（ok / ng）。測る前に、図入門2-1 と見比べて決めておく。分からなければ空欄
# 下の見出し行は消さない。値はすべて自分の測定値を書く
item,kind,rep,ohm,visual

In [ ]:
syn.append_csv("intro2/joints.csv", syn.joints(), ["item", "kind", "rep", "ohm", "visual"])
print(syn.TRUTH["joints"])

In [ ]:
%%writefile intro2/joint_analysis.py
#!/usr/bin/env python3
"""入門2：はんだ付けの検査データ（joints.csv）を解析し、外れ値（はんだ不良の候補）を探す

使い方: cd ~/home-lab/intro2 && uv run python joint_analysis.py joints.csv --res 0.1
  --res : テスタの抵抗レンジの分解能 [Ω]（表示の最小のけた。DE-200A の取扱説明書で要確認）
  --i2c : i2c_check.py の RESULT 行（例: --i2c "RESULT,0x76,1000,0" "RESULT,0x40,1000,0"）

やること
  1. テスト棒どうしを当てたときの値（ゼロ点）のばらつきを調べる
  2. 各リードの値（3回の中央値）からゼロ点を引く
  3. 平均±3標準偏差の方法と、中央値と MAD（中央絶対偏差）を使う頑健な方法で外れ値を探し、比べる
  4. 当て直しによるばらつき（繰り返し誤差）と、リードどうしの差を一元配置分散分析で比べる
  5. I2C のくり返しテストの失敗率を区間推定する
"""
import argparse
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

Z_ROBUST = 3.5      # 頑健な z の基準（Iglewicz と Hoaglin の目安）
Z_CLASSIC = 3.0     # 平均±3標準偏差


def load(path):
    df = pd.read_csv(path, comment="#", skipinitialspace=True, dtype={"ohm": str, "visual": str})
    df.columns = [c.strip() for c in df.columns]
    need = ["item", "kind", "rep", "ohm"]
    miss = [c for c in need if c not in df.columns]
    if miss:
        sys.exit(f"列が足りません: {miss}")
    df = df.dropna(subset=["item", "ohm"]).copy()
    if df.empty:
        sys.exit("データの行がありません。測った値を joints.csv に書き込んでください")
    o = df["ohm"].astype(str).str.strip().str.upper()
    df["open"] = o.isin(["OL", "O.L", "OVER", "INF"])
    df["ohm_v"] = np.where(df["open"], np.inf, pd.to_numeric(o.where(~df["open"]), errors="coerce"))
    if df["ohm_v"].isna().any():
        bad = df.loc[df["ohm_v"].isna(), "ohm"].tolist()
        sys.exit(f"数値として読めない値があります: {bad}")
    if "visual" not in df.columns:
        df["visual"] = np.nan
    return df


def robust_z(x, res):
    """中央値と MAD による z。MAD が分解能より小さいときは分解能を下限にする（量子化対策）"""
    med = np.median(x)
    mad = 1.4826 * np.median(np.abs(x - med))      # 正規分布なら標準偏差に一致するように1.4826倍
    scale = max(mad, res)
    return (x - med) / scale, med, mad, scale


def anova_icc(g):
    """一元配置（変量効果）：当て直しの誤差と、リードどうしの差の大きさを比べる"""
    groups = [v.to_numpy() for _, v in g.groupby("item")["ohm_v"] if len(v) >= 2]
    if len(groups) < 3:
        return None
    k = np.mean([len(v) for v in groups])
    allv = np.concatenate(groups)
    gm = allv.mean()
    ssb = sum(len(v) * (v.mean() - gm) ** 2 for v in groups)
    ssw = sum(((v - v.mean()) ** 2).sum() for v in groups)
    dfb, dfw = len(groups) - 1, len(allv) - len(groups)
    msb, msw = ssb / dfb, ssw / dfw if dfw > 0 else np.nan
    icc = (msb - msw) / (msb + (k - 1) * msw) if msw > 0 else np.nan
    return msb, msw, icc, len(groups)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv")
    ap.add_argument("--res", type=float, default=0.1, help="抵抗レンジの分解能 [Ω]")
    ap.add_argument("--i2c", nargs="*", default=[], help='i2c_check.py の RESULT 行')
    a = ap.parse_args()
    df = load(a.csv)

    # ---------- 1. ゼロ点 ----------
    z = df[(df["kind"] == "zero") & ~df["open"]]["ohm_v"].to_numpy()
    if len(z) >= 3:
        z0 = float(np.median(z))
        print(f"[ゼロ点] テスト棒どうし {len(z)} 回: 中央値 {z0:.3f} Ω、最小 {z.min():.3f}、最大 {z.max():.3f}、"
              f"標準偏差 {z.std(ddof=1):.3f} Ω")
        print("  ゼロ点のばらつきより小さい差は、このテスタでは区別できない")
    else:
        z0 = 0.0
        print("[ゼロ点] zero の行が3回分ないので、ゼロ点は 0 Ω として計算します")

    # ---------- 2・3. 外れ値 ----------
    items = df[df["kind"] != "zero"]
    if items.empty:
        sys.exit("zero 以外の行がありません")
    per = (items.groupby(["kind", "item"])
           .agg(med=("ohm_v", "median"), n=("ohm_v", "size"), n_open=("open", "sum"),
                visual=("visual", lambda v: next((s for s in v if isinstance(s, str)), "")))
           .reset_index())
    per["corr"] = per["med"] - z0
    rows = []
    for kind, g in per.groupby("kind"):
        fin = g[np.isfinite(g["corr"])]
        x = fin["corr"].to_numpy()
        print(f"\n[{kind}] {len(g)} 個（そのうち OL が {int((~np.isfinite(g['corr'])).sum())} 個）")
        if len(x) >= 5:
            rz, med, mad, scale = robust_z(x, a.res)
            mu, sd = x.mean(), x.std(ddof=1)
            cz = (x - mu) / sd if sd > 0 else np.zeros_like(x)
            print(f"  頑健な方法: 中央値 {med:.3f} Ω、MAD×1.4826 = {mad:.3f} Ω"
                  f"{'（分解能より小さいので ' + format(a.res, 'g') + ' Ω を使う）' if mad < a.res else ''}")
            print(f"  平均±3SD : 平均 {mu:.3f} Ω、標準偏差 {sd:.3f} Ω（外れ値があると両方が引っぱられる）")
            fin = fin.assign(rz=rz, cz=cz)
        else:
            print("  5個未満なので、外れ値の判定は省略（値だけ表示）")
            fin = fin.assign(rz=np.nan, cz=np.nan)
        for _, r in g[~np.isfinite(g["corr"])].iterrows():
            rows.append((kind, r["item"], np.inf, np.nan, np.nan, "開放（OL）", r["visual"]))
        for _, r in fin.iterrows():
            flag = []
            if np.isfinite(r["rz"]) and abs(r["rz"]) > Z_ROBUST:
                flag.append("頑健")
            if np.isfinite(r["cz"]) and abs(r["cz"]) > Z_CLASSIC:
                flag.append("3SD")
            rows.append((kind, r["item"], r["corr"], r["rz"], r["cz"], "・".join(flag), r["visual"]))
    res = pd.DataFrame(rows, columns=["kind", "item", "ohm_minus_zero", "robust_z", "classic_z", "flag", "visual"])
    print("\n外れ値の候補（flag が空でないもの。頑健 = |頑健なz| > 3.5、3SD = |平均からのz| > 3）")
    cand = res[res["flag"] != ""]
    print(cand.to_string(index=False, float_format=lambda v: f"{v:.3f}") if len(cand) else "  なし")
    only_r = res[res["flag"] == "頑健"]
    if len(only_r):
        print(f"\n・頑健な方法だけが見つけたもの {len(only_r)} 個：外れ値が標準偏差を大きくして、"
              "平均±3SD では隠れてしまった（マスキング）")

    # 目視との照合
    v = res[res["visual"].isin(["ok", "ng"])]
    if len(v):
        print("\n[目視との照合]（行 = 目視、列 = 測定で候補になったか）")
        print(pd.crosstab(v["visual"], v["flag"] != "", rownames=["目視"], colnames=["測定で候補"]).to_string())
        print("  目視で ng なのに測定で候補にならないものは、テスタでは見つからない不良（冷えたはんだなど）")

    # ---------- 4. 繰り返し誤差とリードの差 ----------
    lead = items[(items["kind"] == "lead") & ~items["open"]]
    flagged = set(res.loc[res["flag"] != "", "item"])
    for title, g in [("候補も含めた全部", lead), ("外れ値の候補を除いた良品だけ", lead[~lead["item"].isin(flagged)])]:
        r = anova_icc(g)
        if not r:
            continue
        msb, msw, icc, ng = r
        print(f"\n[当て直しの誤差とリードの差]（lead・{title}、{ng} 本）")
        print(f"  当て直しによる標準偏差 {np.sqrt(msw):.3f} Ω、級内相関 ICC = {icc:.2f}")
    print("  ICC が 1 に近い = リードどうしの差が当て直しの誤差より大きく、測れている。")
    print("  良品だけで ICC が 0 に近い（推定値なので負になることもある）なら、良いはんだ付けどうしの差は"
          "このテスタでは測れない（mΩ 以下のため）。")

    # ---------- 5. I2C ----------
    for line in a.i2c:
        p = line.split(",")
        if len(p) != 4 or p[0] != "RESULT":
            print("RESULT 行の形が違います:", line)
            continue
        addr, n, k = p[1], int(p[2]), int(p[3])
        lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
        print(f"\n[I2C {addr}] {n} 回中 {k} 回失敗：失敗率 {k/n*100:.3f} %（95% Wilson区間 {lo*100:.3f}〜{hi*100:.3f} %）")
        if k == 0:
            print(f"  0回でも「失敗率が0」とは言えない。上限の目安は 3/n = {3/n*100:.2f} %（3の法則）")

    # ---------- 図 ----------
    fig, ax = plt.subplots(figsize=(9, 4.5))
    kinds = list(res["kind"].unique())
    fin_all = res[np.isfinite(res["ohm_minus_zero"])]
    data = [fin_all.loc[fin_all["kind"] == k, "ohm_minus_zero"].to_numpy() for k in kinds]
    ax.boxplot(data, whis=1.5, showfliers=False)
    ax.set_xticks(range(1, len(kinds) + 1), kinds)
    rng = np.random.default_rng(0)
    top = np.nanmax(fin_all["ohm_minus_zero"]) if len(fin_all) else 1.0
    for i, k in enumerate(kinds, start=1):
        g = res[res["kind"] == k]
        for _, rr in g.iterrows():
            y = rr["ohm_minus_zero"] if np.isfinite(rr["ohm_minus_zero"]) else top * 1.15 + 0.1
            c = "tab:red" if rr["flag"] else "tab:blue"
            m = "x" if not np.isfinite(rr["ohm_minus_zero"]) else "o"
            xx = i + rng.uniform(-0.12, 0.12)
            ax.plot(xx, y, m, color=c, ms=5, alpha=0.8)
            if rr["flag"]:
                ax.annotate(str(rr["item"]), (xx, y), textcoords="offset points", xytext=(5, 2), fontsize=8)
    ax.set_ylabel("抵抗 − ゼロ点 [Ω]（× は OL）")
    ax.set_title("はんだ付けの検査（赤 = 外れ値の候補）")
    ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig("joints.png", dpi=150)
    print("\njoints.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
out_j, rc = run('joint_analysis.py joints.csv --res 0.1 --i2c "RESULT,0x40,1000,0" "RESULT,0x76,1000,0"', cwd="intro2")
show("intro2/joints.png", width=700)

In [ ]:
# 確かめ（入門2-7 の結果の例）
ok(rc == 0, "joint_analysis.py がエラーなく終わった")
z0 = num(r"中央値 ([\d.]+) Ω、最小", out_j)
ok(0.1 <= z0 <= 0.5, f"ゼロ点は 0.1〜0.5 Ω 程度（{z0} Ω）")
ok(re.search(r"L07 .*\s頑健\s", out_j) is not None, "1.5 Ω 足した L07 は、頑健な方法だけが見つける（マスキング）")
ok(re.search(r"L15 .*頑健・3SD", out_j) is not None, "3.1 Ω 足した L15 は、両方の方法が見つける")
ok(re.search(r"L20 .*開放（OL）", out_j) is not None, "L20 は開放（OL）")
ok("マスキング" in out_j, "「頑健な方法だけが見つけたもの」の説明が出る")
iccs = [float(v) for v in re.findall(r"ICC = (-?[\d.]+)", out_j)]
ok(iccs[0] > 0.9 and abs(iccs[1]) < 0.4, f"ICC：外れ値を含めると {iccs[0]}（1 に近い）、良品だけだと {iccs[1]}（0 付近）")
ok(re.search(r"ng +2 +3", out_j) is not None, "目視で ng の5本のうち、測定で候補になるのは3本だけ（冷えたはんだ2本は導通する）")
up = num(r"Wilson区間 0.000〜([\d.]+) %", out_j)
ok(abs(up - 0.38) < 0.01 and "3/n = 0.30 %" in out_j, f"1000 回中 0 回失敗：95% 上限 約 {up}%（Wilson）、3の法則で 0.3%")

## 5. 入門3 の数値例と、BME280 の補正式の検算（入門3-3、手順2）

I2C のプルアップの立ち上がり時間 $t_r \approx 0.8473R_pC_b$、プルアップの下限、INA219 の校正値 $\mathrm{Cal} = 0.04096/(\mathrm{Current\_LSB}\times R_{\mathrm{shunt}})$、自己相関があるときの有効サンプル数 $n_{\mathrm{eff}} \approx n(1-\rho)/(1+\rho)$、白色雑音の平均の効果を計算します。

In [ ]:
tr = lambda Rp, Cb=100e-12: Rp * Cb * np.log((1 - 0.3) / (1 - 0.7))
for Rp in (2.2e3, 5e3, 10e3):
    print(f"R_p = {Rp/1e3:.1f} kΩ：t_r = {tr(Rp)*1e9:.0f} ns")
Rp_min = (3.3 - 0.4) / 3e-3
cal = int(0.04096 / (1e-4 * 0.1))
r09, r997 = (1 + 0.9) / (1 - 0.9), (1 + 0.997) / (1 - 0.997)
print(f"R_p の下限 {Rp_min:.0f} Ω、Cal = {cal}、ρ = 0.9 で n/{r09:.0f}、ρ = 0.997 で n/{r997:.0f}（区間は {np.sqrt(r997):.1f} 倍）、"
      f"1.5 Pa を 50 回平均 → {1.5/np.sqrt(50):.2f} Pa")
ok(abs(np.log(0.7 / 0.3) - 0.8473) < 1e-4, "ln((1−0.3)/(1−0.7)) ≈ 0.8473")
ok([round(tr(r) * 1e9) for r in (2.2e3, 5e3, 10e3)] == [186, 424, 847], "t_r は 2.2 kΩ で 186 ns、5 kΩ で 424 ns、10 kΩ で 847 ns（図入門3-2 (b)）")
ok(tr(10e3) < 1000e-9 and tr(10e3) > 300e-9, "10 kΩ・100 pF は 100 kHz（1000 ns）なら使えるが、400 kHz（300 ns）には遅すぎる")
ok(abs(Rp_min - 1000) < 50, "R_p ≳ 2.9 V / 3 mA ≈ 1 kΩ")
ok(cal == 4096, "Current_LSB 0.1 mA・0.1 Ω で Cal = 4096")
ok(round(r09) == 19 and abs(r997 - 670) < 10 and round(np.sqrt(r997)) == 26 and abs(6000 / r997 - 9) < 0.5,
   "n_eff：ρ = 0.9 で n/19、ρ = 0.997 で約 n/670（6000 点が約9点分、区間は約26倍）")
ok(abs(1.5 / np.sqrt(50) - 0.21) < 0.005, "雑音 1.5 Pa を 50 回（1 秒）平均すると約 0.21 Pa")

BME280・INA219 のドライバ（MicroPython 用ですが、`machine` を読み込まないので Mac の Python でも import できます）と、補正式の検算 `test_bme280_vector.py` を、本文と同じ `lib/pico/`・`lib/mac/` に書き出して実行します（手順2。Pico なしで Mac だけで動く部分）。BMP280 のデータシートの計算例の値は、本文と同じく「データシートで要確認」です。

In [ ]:
%%writefile lib/pico/bme280.py
# BME280 / BMP280 ドライバ（MicroPython 用。補正式のテストのため CPython でも import できる）
# 置き場所: Mac の ~/home-lab/lib/pico/bme280.py が正本 → Pico には :lib/bme280.py としてコピーする
# 使い方:
#   from machine import I2C, Pin
#   from bme280 import BME280
#   i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000)
#   s = BME280(i2c)              # アドレスは 0x76 → 0x77 の順に探す
#   t, p, h = s.read()           # 温度 [°C]、気圧 [Pa]、湿度 [%RH]（BMP280 なら h は None）
#   print(s.name, s.chip_id)     # "BME280" と 0x60、または "BMP280" と 0x58
# 補正式は Bosch のデータシートにある整数版の式をそのまま書いたもの（式の番号・節はデータシートで要確認）。
import struct
import time

REG_CALIB_TP = 0x88      # 0x88〜0x9F：温度・気圧の補正係数（BME280・BMP280 共通）
REG_CALIB_H1 = 0xA1      # 湿度の補正係数（BME280 だけ）
REG_CALIB_H2 = 0xE1      # 0xE1〜0xE7
REG_ID = 0xD0
REG_RESET = 0xE0
REG_CTRL_HUM = 0xF2      # BME280 だけ
REG_STATUS = 0xF3
REG_CTRL_MEAS = 0xF4
REG_CONFIG = 0xF5
REG_DATA = 0xF7          # 0xF7〜0xFE（BMP280 は 0xF7〜0xFC）

CHIP_BME280 = 0x60
CHIP_BMP280 = (0x56, 0x57, 0x58)     # 0x58 が量産品。0x56・0x57 は試作品の値とされる（要確認）
OSRS_CODE = {0: 0, 1: 1, 2: 2, 4: 3, 8: 4, 16: 5}   # 平均回数（オーバーサンプリング）→ レジスタの値
IIR_CODE = {0: 0, 2: 1, 4: 2, 8: 3, 16: 4}


def _sleep_ms(ms):
    f = getattr(time, "sleep_ms", None)
    if f:
        f(ms)
    else:
        time.sleep(ms / 1000)


class BME280:
    def __init__(self, i2c, addr=None, osrs_t=2, osrs_p=16, osrs_h=1, iir=0):
        self.i2c = i2c
        if addr is None:
            found = i2c.scan()
            for a in (0x76, 0x77):
                if a in found:
                    addr = a
                    break
            else:
                raise OSError("BME280/BMP280 が 0x76・0x77 に見つかりません")
        self.addr = addr
        self.chip_id = self._r8(REG_ID)
        if self.chip_id == CHIP_BME280:
            self.name, self.has_humidity = "BME280", True
        elif self.chip_id in CHIP_BMP280:
            self.name, self.has_humidity = "BMP280", False
        else:
            raise OSError("チップID 0x%02X は BME280（0x60）でも BMP280（0x58）でもありません" % self.chip_id)
        self._w8(REG_RESET, 0xB6)              # ソフトリセット
        _sleep_ms(5)
        for _ in range(20):                    # 補正係数の読み込み（im_update）が終わるまで待つ
            if not self._r8(REG_STATUS) & 0x01:
                break
            _sleep_ms(2)
        self._read_calibration()
        self.configure(osrs_t, osrs_p, osrs_h, iir)

    # ---------------------------------------------------------------- レジスタの読み書き
    def _r8(self, reg):
        return self.i2c.readfrom_mem(self.addr, reg, 1)[0]

    def _w8(self, reg, val):
        self.i2c.writeto_mem(self.addr, reg, bytes([val & 0xFF]))

    def _read_calibration(self):
        c = self.i2c.readfrom_mem(self.addr, REG_CALIB_TP, 24)
        (self.T1, self.T2, self.T3,
         self.P1, self.P2, self.P3, self.P4, self.P5,
         self.P6, self.P7, self.P8, self.P9) = struct.unpack("<HhhHhhhhhhhh", c)
        if self.has_humidity:
            self.H1 = self._r8(REG_CALIB_H1)
            e = self.i2c.readfrom_mem(self.addr, REG_CALIB_H2, 7)
            self.H2 = struct.unpack("<h", e[0:2])[0]
            self.H3 = e[2]
            h4 = (e[3] << 4) | (e[4] & 0x0F)          # 12ビットの符号つき整数
            h5 = (e[5] << 4) | (e[4] >> 4)
            self.H4 = h4 - 4096 if h4 & 0x800 else h4
            self.H5 = h5 - 4096 if h5 & 0x800 else h5
            self.H6 = struct.unpack("<b", e[6:7])[0]

    def configure(self, osrs_t=2, osrs_p=16, osrs_h=1, iir=0):
        """平均回数（1, 2, 4, 8, 16。0 は測らない）と IIR フィルタ（0 = なし）を設定する"""
        self._ctrl_meas = (OSRS_CODE[osrs_t] << 5) | (OSRS_CODE[osrs_p] << 2)   # 最後の2ビット = モード
        if self.has_humidity:
            self._w8(REG_CTRL_HUM, OSRS_CODE[osrs_h])     # ctrl_meas を書いたときに有効になる
        self._w8(REG_CONFIG, IIR_CODE[iir] << 2)
        self._w8(REG_CTRL_MEAS, self._ctrl_meas)          # スリープモード

    # ---------------------------------------------------------------- 測定
    def read_raw(self):
        """強制モードで1回測り、補正前の値（adc_T, adc_P, adc_H）を返す"""
        self._w8(REG_CTRL_MEAS, self._ctrl_meas | 0x01)   # 強制モード：1回測ってスリープに戻る
        _sleep_ms(2)
        for _ in range(100):                               # measuring ビットが 0 になるまで待つ
            if not self._r8(REG_STATUS) & 0x08:
                break
            _sleep_ms(1)
        n = 8 if self.has_humidity else 6
        d = self.i2c.readfrom_mem(self.addr, REG_DATA, n)
        adc_p = (d[0] << 12) | (d[1] << 4) | (d[2] >> 4)
        adc_t = (d[3] << 12) | (d[4] << 4) | (d[5] >> 4)
        adc_h = (d[6] << 8) | d[7] if self.has_humidity else None
        return adc_t, adc_p, adc_h

    def read(self):
        """(温度 [°C], 気圧 [Pa], 湿度 [%RH] または None) を返す"""
        return self.compensate(*self.read_raw())

    # ---------------------------------------------------------------- 補正式（データシートの整数版）
    def compensate(self, adc_t, adc_p, adc_h=None):
        # 温度：t_fine は気圧・湿度の補正にも使う
        var1 = (((adc_t >> 3) - (self.T1 << 1)) * self.T2) >> 11
        var2 = (((((adc_t >> 4) - self.T1) * ((adc_t >> 4) - self.T1)) >> 12) * self.T3) >> 14
        t_fine = var1 + var2
        t_c = ((t_fine * 5 + 128) >> 8) / 100.0            # 0.01 °C 単位 → °C

        # 気圧（64ビット版。Python の整数は桁あふれしない）
        p_pa = None
        if adc_p != 0x80000:                               # 0x80000 は「測っていない」
            var1 = t_fine - 128000
            var2 = var1 * var1 * self.P6
            var2 = var2 + ((var1 * self.P5) << 17)
            var2 = var2 + (self.P4 << 35)
            var1 = ((var1 * var1 * self.P3) >> 8) + ((var1 * self.P2) << 12)
            var1 = (((1 << 47) + var1) * self.P1) >> 33
            if var1 != 0:
                p = 1048576 - adc_p
                p = (((p << 31) - var2) * 3125) // var1
                var1 = (self.P9 * (p >> 13) * (p >> 13)) >> 25
                var2 = (self.P8 * p) >> 19
                p = ((p + var1 + var2) >> 8) + (self.P7 << 4)
                p_pa = p / 256.0                            # Q24.8 形式 → Pa

        # 湿度（32ビット版）。BMP280 には湿度センサがない
        h_pct = None
        if self.has_humidity and adc_h is not None and adc_h != 0x8000:
            v = t_fine - 76800
            v = (((((adc_h << 14) - (self.H4 << 20) - (self.H5 * v)) + 16384) >> 15) *
                 (((((((v * self.H6) >> 10) * (((v * self.H3) >> 11) + 32768)) >> 10) + 2097152) *
                   self.H2 + 8192) >> 14))
            v = v - (((((v >> 15) * (v >> 15)) >> 7) * self.H1) >> 4)
            v = 0 if v < 0 else v
            v = 419430400 if v > 419430400 else v
            h_pct = (v >> 12) / 1024.0
        return t_c, p_pa, h_pct

In [ ]:
%%writefile lib/pico/ina219.py
# INA219 ドライバ（MicroPython 用）
# 置き場所: Mac の ~/home-lab/lib/pico/ina219.py が正本 → Pico には :lib/ina219.py としてコピーする
# 使い方:
#   from machine import I2C, Pin
#   from ina219 import INA219
#   i2c = I2C(0, sda=Pin(4), scl=Pin(5), freq=100_000)
#   m = INA219(i2c)                      # 0x40、シャント 0.1Ω、電流の1目盛り 0.1mA、±320mV、平均なし
#   bus_v, shunt_mv, current_ma = m.read()
# レジスタの番地・ビットの意味・変換時間は TI の INA219 データシートで要確認。
import time

REG_CONFIG = 0x00
REG_SHUNT = 0x01        # シャント電圧（符号つき、1目盛り 10µV）
REG_BUS = 0x02          # バス電圧（ビット15〜3、1目盛り 4mV。ビット0 = OVF）
REG_POWER = 0x03
REG_CURRENT = 0x04
REG_CALIB = 0x05

PGA_CODE = {40: 0, 80: 1, 160: 2, 320: 3}          # シャント電圧の範囲 ±mV → PG ビット
AVG_CODE = {1: 0b0011, 2: 0b1001, 4: 0b1010, 8: 0b1011,
            16: 0b1100, 32: 0b1101, 64: 0b1110, 128: 0b1111}   # 12ビットで n 回平均
CONV_MS = {1: 0.532, 2: 1.06, 4: 2.13, 8: 4.26, 16: 8.51, 32: 17.02, 64: 34.05, 128: 68.10}


def _s16(b):
    v = (b[0] << 8) | b[1]
    return v - 65536 if v & 0x8000 else v


class INA219:
    def __init__(self, i2c, addr=0x40, r_shunt=0.1, i_lsb=1e-4, range_mv=320, avg=1, bus_32v=False):
        self.i2c = i2c
        self.addr = addr
        self.r_shunt = r_shunt          # シャント抵抗 [Ω]（モジュールの R100 = 0.1Ω。実物で要確認）
        self.i_lsb = i_lsb              # 電流レジスタの1目盛り [A]
        self.range_mv = range_mv
        self.avg = avg
        self.conv_ms = CONV_MS[avg]     # 1回の変換時間 [ms]（シャント・バスそれぞれ）
        self._w16(REG_CONFIG, 0x8000)   # リセット
        time.sleep_ms(1) if hasattr(time, "sleep_ms") else time.sleep(0.001)
        cfg = ((1 if bus_32v else 0) << 13) | (PGA_CODE[range_mv] << 11) \
            | (AVG_CODE[avg] << 7) | (AVG_CODE[avg] << 3) | 0b111   # シャントとバスを連続で測る
        self._w16(REG_CONFIG, cfg)
        self.config = cfg
        # キャリブレーション：Cal = trunc(0.04096 / (電流の1目盛り × シャント抵抗))
        self.cal = int(0.04096 / (i_lsb * r_shunt))
        self._w16(REG_CALIB, self.cal)

    def _w16(self, reg, v):
        self.i2c.writeto_mem(self.addr, reg, bytes([(v >> 8) & 0xFF, v & 0xFF]))

    def _r(self, reg):
        return self.i2c.readfrom_mem(self.addr, reg, 2)

    def shunt_mv(self):
        return _s16(self._r(REG_SHUNT)) * 0.01            # 10µV = 0.01mV

    def bus_v(self):
        b = self._r(REG_BUS)
        raw = (b[0] << 8) | b[1]
        self.overflow = bool(raw & 0x01)                  # OVF：電流・電力の計算があふれた
        return (raw >> 3) * 0.004

    def current_ma(self):
        """電流レジスタの値 [mA]（キャリブレーションを使ってチップの中で計算されたもの）"""
        return _s16(self._r(REG_CURRENT)) * self.i_lsb * 1000

    def read(self):
        """(バス電圧 [V]、シャント電圧 [mV]、電流 [mA]) を返す。電流はシャント電圧 ÷ シャント抵抗で求める"""
        sh = self.shunt_mv()
        return self.bus_v(), sh, sh / self.r_shunt

In [ ]:
%%writefile lib/mac/test_bme280_vector.py
#!/usr/bin/env python3
"""bme280.py の補正式を確かめる（Pico なしで Mac だけで動く）

使い方: cd ~/home-lab && uv run python lib/mac/test_bme280_vector.py
  1. 温度・気圧：BMP280 データシートの計算例と比べる
     計算例の値（補正係数・adc の値・答え）は BMP280 データシートの「補正式の計算例」の表で要確認。
  2. 湿度：データシートには湿度の計算例がないので、整数版の式（bme280.py）を、
     浮動小数点版の式（BME280 データシートの付録にある式。式の形はデータシートで要確認）と比べる。
     補正係数 H4・H5 は 12ビットの符号つき整数で、レジスタの読み方をまちがえやすいので、
     負の値を含む係数の組でも試す。
"""
import pathlib
import struct
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parent.parent / "pico"))
import bme280  # noqa: E402

CALIB = dict(T1=27504, T2=26435, T3=-1000, P1=36477, P2=-10685, P3=3024, P4=2855,
             P5=140, P6=-7, P7=15500, P8=-14600, P9=6000)
ADC_T, ADC_P = 519888, 415148
EXPECT_T, EXPECT_P = 25.08, 100653.27      # データシートの答え（°C、Pa）

# 湿度の補正係数の組 (H1, H2, H3, H4, H5, H6) と、試す adc_H の値。
# 1つ目は実物によくある大きさの値、2つ目は H4・H5 が負、3つ目は H6 が負（どれも作った値）
HUM_CASES = [((75, 362, 0, 313, 50, 30), (24000, 27000, 32000)),
             ((75, 370, 0, -10, -50, 30), (2000, 6000, 10000)),
             ((0, 350, 10, 300, 20, -20), (20000, 27000, 32000))]
HUM_TOL = 0.05                              # 整数版と浮動小数点版の差の許容 [%RH]


class FakeI2C:
    """BMP280 / BME280 のふりをする偽の I2C（チップID と補正係数だけを返す）"""
    def __init__(self, hum=None):
        chip = 0x58 if hum is None else 0x60
        self.mem = {0xD0: bytes([chip]), 0xF3: bytes([0])}
        c = struct.pack("<HhhHhhhhhhhh", *[CALIB[k] for k in
                        ["T1", "T2", "T3", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9"]])
        self.mem[0x88] = c
        if hum is not None:
            h1, h2, h3, h4, h5, h6 = hum
            h4 &= 0xFFF                    # 12ビットの2の補数にする
            h5 &= 0xFFF
            self.mem[0xA1] = bytes([h1])
            # 0xE4 = H4 の上位8ビット、0xE5 = 下位4ビットが H4・上位4ビットが H5、0xE6 = H5 の上位8ビット
            self.mem[0xE1] = (struct.pack("<hB", h2, h3)
                              + bytes([h4 >> 4, (h4 & 0x0F) | ((h5 & 0x0F) << 4), h5 >> 4])
                              + struct.pack("<b", h6))

    def scan(self):
        return [0x76]

    def readfrom_mem(self, addr, reg, n):
        return self.mem.get(reg, bytes(n))[:n].ljust(n, b"\0")

    def writeto_mem(self, addr, reg, data):
        pass


def t_fine_of(s, adc_t):
    """温度の補正の途中の値 t_fine（bme280.py の compensate と同じ式）"""
    var1 = (((adc_t >> 3) - (s.T1 << 1)) * s.T2) >> 11
    var2 = (((((adc_t >> 4) - s.T1) * ((adc_t >> 4) - s.T1)) >> 12) * s.T3) >> 14
    return var1 + var2


def hum_float(t_fine, adc_h, h1, h2, h3, h4, h5, h6):
    """湿度の浮動小数点版の補正式 [%RH]"""
    v = t_fine - 76800.0
    v = (adc_h - (h4 * 64.0 + h5 / 16384.0 * v)) * \
        (h2 / 65536.0 * (1.0 + h6 / 67108864.0 * v * (1.0 + h3 / 67108864.0 * v)))
    v = v * (1.0 - h1 * v / 524288.0)
    return min(max(v, 0.0), 100.0)


# 1. 温度・気圧（BMP280 の計算例）
s = bme280.BME280(FakeI2C())
t, p, h = s.compensate(ADC_T, ADC_P)
print(f"{s.name}（チップID 0x{s.chip_id:02X}）")
print(f"温度 = {t:.2f} °C（計算例 {EXPECT_T} °C）")
print(f"気圧 = {p:.2f} Pa（計算例 {EXPECT_P} Pa）")
print(f"湿度 = {'なし' if h is None else h}")
ok_tp = abs(t - EXPECT_T) < 0.005 and abs(p - EXPECT_P) < 0.5
print("OK：温度・気圧の補正式はデータシートの計算例と一致" if ok_tp else "NG：補正式か計算例の値を見直す")

# 2. 湿度（整数版と浮動小数点版の比較）
ok_h = True
worst = 0.0
for hum, adcs in HUM_CASES:
    s = bme280.BME280(FakeI2C(hum))
    got = (s.H1, s.H2, s.H3, s.H4, s.H5, s.H6)
    if got != hum:
        print(f"NG：補正係数の読み方がおかしい（入れた値 {hum}、読んだ値 {got}）")
        ok_h = False
        continue
    for adc_t in (480000, ADC_T, 560000):
        for adc_h in adcs:
            t, p, h = s.compensate(adc_t, ADC_P, adc_h)
            hf = hum_float(t_fine_of(s, adc_t), adc_h, *hum)
            worst = max(worst, abs(h - hf))
            if abs(h - hf) > HUM_TOL:
                print(f"NG：係数 {hum}、{t:.2f} °C、adc_H = {adc_h}：整数版 {h:.3f} %RH、浮動小数点版 {hf:.3f} %RH")
                ok_h = False
print(f"{s.name}（チップID 0x{s.chip_id:02X}）：湿度 {len(HUM_CASES)} 組の係数 × 9 点、"
      f"整数版と浮動小数点版の差は最大 {worst:.3f} %RH")
print("OK：湿度の補正式は浮動小数点版と一致" if ok_h else "NG：湿度の補正式か、補正係数の読み方を見直す")
sys.exit(0 if (ok_tp and ok_h) else 1)

In [ ]:
out_v, rc = run("lib/mac/test_bme280_vector.py")
ok(rc == 0, "test_bme280_vector.py が終了コード 0 で終わった（温度・気圧・湿度とも OK）")
ok("OK：温度・気圧の補正式はデータシートの計算例と一致" in out_v, "温度 25.08 °C・気圧 100653.27 Pa（計算例）と一致")
ok("OK：湿度の補正式は浮動小数点版と一致" in out_v, "湿度の整数版と浮動小数点版の差が 0.05 %RH 以内（H4・H5 が負の組も含む）")

INA219 のドライバも、偽の I2C（書き込んだレジスタを覚えておくだけ）で、校正値 4096 を書くことと、シャント電圧・バス電圧のレジスタの読み方を確かめておきます（ノートブック版で追加した確かめです）。

In [ ]:
sys.path.insert(0, os.path.join(HOME, "lib/pico"))
import ina219


class FakeI2C:
    def __init__(self):
        self.mem = {}
    def writeto_mem(self, addr, reg, data):
        self.mem[reg] = bytes(data)
    def readfrom_mem(self, addr, reg, n):
        return self.mem.get(reg, bytes(n))


bus = FakeI2C()
m = ina219.INA219(bus)
cal_reg = int.from_bytes(bus.mem[ina219.REG_CALIB], "big")
bus.mem[ina219.REG_SHUNT] = (40).to_bytes(2, "big", signed=True)          # 40 × 10 µV = 0.40 mV
bus.mem[ina219.REG_BUS] = ((788 << 3) | 0b10).to_bytes(2, "big")           # 788 × 4 mV = 3.152 V
print("Cal レジスタ", cal_reg, "／ read() =", m.read())
ok(cal_reg == 4096, "INA219 の校正レジスタに 4096 が書かれる")
bv, sh, cur = m.read()
ok(abs(sh - 0.40) < 1e-9 and abs(bv - 3.152) < 1e-9 and abs(cur - 4.0) < 1e-9,
   "シャント 0.40 mV → 4.0 mA（0.1 Ω）、バス電圧 3.152 V（LED で約 4 mA、約 0.4 mV、3.1〜3.2 V の目安どおり）")

## 6. 静止状態の雑音（入門3 の雑音の記録と `analyze_noise.py`）

`serial_logger.py` が保存する CSV（列 `host_time,t_ms,temp_c,press_pa,hum_pct,pico_c,bus_v,shunt_mv,current_ma`）と同じ形のものを、`logger.py` の `MODE = "noise"`（平均なし、20 ms ごと、120 秒）のつもりで合成します。

- 温度：ゆっくり動く揺らぎ（AR(1)、$\rho = 0.999$）＋小さな白色雑音、0.01 °C 刻み。
- 気圧：白色雑音 1.5 Pa ＋ ゆっくりした揺らぎ（ランダムウォーク）。アラン偏差が数秒で下がらなくなるように選んだ。
- 電流：LED 点けっぱなしで 4.0 mA、0.1 mA 刻み（シャント 0.1 Ω、10 µV 刻み）。
- PWM 50%（1 kHz）：INA219 が1回の変換（平均回数1 で 0.532 ms、128 で 68.1 ms）のあいだの平均を読むとみなし、PWM とは同期していない（読み始めの位相がばらばら）とした。128 回平均の記録の間隔は `logger.py` と同じ `max(20, int(2.2 × 68.1) + 1)` = 150 ms。

In [ ]:
syn.noise("intro3/data/noise_dc.csv")
syn.noise("intro3/data/noise_pwm1.csv", led="pwm", ina_avg=1, period_ms=20, seed=7)
syn.noise("intro3/data/noise_pwm128.csv", led="pwm", ina_avg=128, period_ms=max(20, int(2.2 * 68.10) + 1), seed=8)
print(pd.read_csv("intro3/data/noise_dc.csv").head())

In [ ]:
%%writefile intro3/analyze_noise.py
#!/usr/bin/env python3
"""入門3：静止状態の雑音を調べる（標準偏差・自己相関・アラン分散）

使い方: cd ~/home-lab/intro3 && uv run python analyze_noise.py data/noise_dc.csv
        uv run python analyze_noise.py data/noise_pwm1.csv data/noise_pwm128.csv --cols current_ma
  --cols : 解析する列（既定は temp_c press_pa current_ma のうち、ファイルにあるもの）

やること（列ごと）
  1. 平均・標準偏差・表示の刻み（いちばん小さい変化）
  2. 自己相関関数：となりどうしの値が似ているか（似ていると「独立なデータ」として扱えない）
  3. 平均の95%信頼区間を、独立とみなした場合と、自己相関を考えた場合（HAC をラグを変えて、
     AR(1) の有効サンプル数で補正）で比べる
  4. アラン偏差：何個ずつ平均すると、どこまでばらつきが減るか
"""
import argparse
import pathlib

import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=FutureWarning)   # statsmodels の将来の仕様変更の予告を表示しない

UNIT = {"temp_c": "°C", "press_pa": "Pa", "current_ma": "mA", "bus_v": "V", "hum_pct": "%RH"}


def allan_dev(y, tau0, n_tau=25):
    """重なりありのアラン偏差。y は等間隔の値、tau0 はその間隔 [s]"""
    y = np.asarray(y, float)
    n = len(y)
    ms = np.unique(np.logspace(0, np.log10(max(1, n // 4)), n_tau).astype(int))
    c = np.concatenate([[0.0], np.cumsum(y)])
    taus, adev = [], []
    for m in ms:
        avg = (c[m:] - c[:-m]) / m                   # 長さ m の移動平均（重なりあり）
        d = avg[m:] - avg[:-m]                       # となりあう（重ならない）平均どうしの差
        if len(d) < 2:
            break
        taus.append(m * tau0)
        adev.append(np.sqrt(0.5 * np.mean(d ** 2)))
    return np.array(taus), np.array(adev)


def mean_ci(y, tau0, rho1):
    """平均の95%区間の半幅を、3つの方法で求める
    ・独立とみなした場合（s/√n をもとにした、ふつうの区間）
    ・HAC（Newey–West）。何個先までの自己相関を考えるか（ラグ）を、秒で決めて変えてみる
    ・AR(1) とみなし、有効サンプル数 n(1−ρ)/(1+ρ) で補正した場合（独立の区間 × √((1+ρ)/(1−ρ))）"""
    X = np.ones((len(y), 1))
    lo, hi = sm.OLS(y, X).fit().conf_int()[0]
    h_naive = (hi - lo) / 2
    hacs = []
    for sec in (1, 5, 20):                           # ラグ = 1秒・5秒・20秒ぶん
        L = int(round(sec / tau0))
        if L < 1 or L > len(y) // 5:                 # 記録の長さの 1/5 を超えるラグは使わない
            continue
        hac = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": L})
        lo2, hi2 = hac.conf_int()[0]
        hacs.append((L, L * tau0, (hi2 - lo2) / 2))
    rho = min(max(rho1, -0.9), 0.999)               # ρ が ±1 に近すぎると式が発散するので抑える
    n_eff = len(y) * (1 - rho) / (1 + rho)
    h_ar1 = h_naive * np.sqrt((1 + rho) / (1 - rho))
    return h_naive, hacs, n_eff, h_ar1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", nargs="+")
    ap.add_argument("--cols", nargs="*", default=["temp_c", "press_pa", "current_ma"])
    a = ap.parse_args()

    dfs = {pathlib.Path(p).stem: pd.read_csv(p) for p in a.csv}
    cols = [c for c in a.cols if any(c in d.columns for d in dfs.values())]
    fig, axes = plt.subplots(3, len(cols), figsize=(5.2 * len(cols), 10), squeeze=False)
    for j, col in enumerate(cols):
        u = UNIT.get(col, "")
        for name, d in dfs.items():
            if col not in d.columns:
                continue
            dd = d[[col, "t_ms"]].dropna()         # 値と時刻を同じ行でそろえて欠けを除く
            y = dd[col].to_numpy(float)
            tau0 = np.median(np.diff(dd["t_ms"].to_numpy())) / 1000.0
            steps = np.diff(np.unique(np.round(y, 6)))
            step = steps.min() if len(steps) else float("nan")
            print(f"\n[{name} / {col}] {len(y)} 点、間隔 {tau0*1000:.1f} ms（{len(y)*tau0:.0f} 秒）")
            print(f"  平均 {y.mean():.4f} {u}、標準偏差 {y.std(ddof=1):.4f} {u}、"
                  f"最小 {y.min():.4f}、最大 {y.max():.4f}、いちばん小さい刻み {step:.4g} {u}")
            if np.ptp(y) == 0:
                print("  値がまったく変わらないので、以下は省略")
                continue
            r, ci = acf(y, nlags=min(50, len(y) // 4), alpha=0.05)
            print(f"  自己相関 ρ(1) = {r[1]:.3f}、ρ(5) = {r[5]:.3f}、ρ(10) = {r[min(10, len(r)-1)]:.3f}")
            h0, hacs, n_eff, h_ar1 = mean_ci(y, tau0, r[1])
            print(f"  平均の95%区間：独立とみなすと ±{h0:.4g} {u}")
            for L, sec, hh in hacs:
                print(f"                  HAC（ラグ {L:5d} 個 = {sec:4.0f} 秒）だと ±{hh:.4g} {u}（{hh/h0:.1f} 倍）")
            print(f"                  AR(1) の補正（有効サンプル数 約 {n_eff:.0f} 点）だと ±{h_ar1:.4g} {u}（{h_ar1/h0:.1f} 倍）")
            taus, ad = allan_dev(y, tau0)
            k = np.argmin(ad)
            print(f"  アラン偏差：τ = {taus[0]:.3g} s で {ad[0]:.4g} {u} → 最小は τ = {taus[k]:.3g} s で {ad[k]:.4g} {u}")
            ax = axes[0, j]
            ax.hist(y, bins=min(60, max(10, len(np.unique(y)))), alpha=0.6, label=name)
            ax.set_title(f"{col} の分布")
            ax.set_xlabel(f"{col} [{u}]")
            ax = axes[1, j]
            ax.stem(np.arange(len(r)), r, basefmt=" ", label=name)
            ax.fill_between(np.arange(len(r)), ci[:, 0] - r, ci[:, 1] - r, alpha=0.15)
            ax.set_title("自己相関（帯は独立なら入る95%範囲）")
            ax.set_xlabel("ずらした個数（ラグ）")
            ax = axes[2, j]
            ax.loglog(taus, ad, "o-", ms=3, label=name)
            ax.loglog(taus, ad[0] * np.sqrt(taus[0] / taus), "--", color="gray", lw=1)
            ax.set_title("アラン偏差（点線は傾き −1/2：白色雑音だけなら）")
            ax.set_xlabel("平均する時間 τ [s]")
            ax.set_ylabel(f"[{u}]")
            ax.grid(True, which="both", alpha=0.3)
        for i in range(3):
            axes[i, j].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("noise.png", dpi=130)
    print("\nnoise.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
out_n, rc = run("analyze_noise.py data/noise_dc.csv", cwd="intro3")
ok(rc == 0, "analyze_noise.py（点けっぱなし）がエラーなく終わった")
show("intro3/noise.png", width=900)

In [ ]:
# 確かめ（入門3-7 の結果の例）。数は analyze_noise.py の関数をそのまま使って計算し直す
sys.path.insert(0, os.path.join(HOME, "intro3"))
import analyze_noise as an
from statsmodels.tsa.stattools import acf
dn = pd.read_csv("intro3/data/noise_dc.csv")
tau0 = 0.020
yT = dn["temp_c"].to_numpy()
r1 = acf(yT, nlags=2)[1]
h0, hacs, n_eff, h_ar1 = an.mean_ci(yT, tau0, r1)
ratio = max(h for _, _, h in hacs) / h0
ok(r1 > 0.9, f"温度は自己相関が強い（ρ(1) = {r1:.3f}）")
ok(3 < ratio, f"温度の平均の区間：HAC（ラグ 20 秒）は独立とみなした区間の {ratio:.1f} 倍（数倍〜十数倍以上）")
yP = dn["press_pa"].to_numpy()
taus, ad = an.allan_dev(yP, tau0)
slope = np.polyfit(np.log(taus[taus <= 0.5]), np.log(ad[taus <= 0.5]), 1)[0]
ad1 = np.interp(1.0, taus, ad)
tmin = taus[np.argmin(ad)]
ok(1.0 <= yP.std(ddof=1) <= 3.0, f"気圧の標準偏差 {yP.std(ddof=1):.2f} Pa（1〜数 Pa）")
ok(-0.6 < slope < -0.4, f"気圧のアラン偏差は 0.5 秒までは傾き {slope:.2f}（−1/2）で下がる")
ok(abs(ad1 - 0.21) < 0.06, f"τ = 1 秒（50 回平均）で {ad1:.3f} Pa（白色雑音 1.5 Pa なら 0.21 Pa）")
ok(2 <= tmin <= 60, f"最小は τ = {tmin:.2g} 秒（数秒〜数十秒で下がらなくなる）")
steps = np.diff(np.unique(dn["current_ma"]))
ok(abs(steps.min() - 0.1) < 1e-6 and dn["current_ma"].std() < 0.2, "電流は 0.1 mA 刻みの1〜2目盛りでばらつく")

In [ ]:
out_p, rc = run("analyze_noise.py data/noise_pwm1.csv data/noise_pwm128.csv --cols current_ma", cwd="intro3")
ok(rc == 0, "analyze_noise.py（PWM 50%）がエラーなく終わった")
show("intro3/noise.png", width=500)
c1 = pd.read_csv("intro3/data/noise_pwm1.csv")["current_ma"]
c128 = pd.read_csv("intro3/data/noise_pwm128.csv")["current_ma"]
ok(c1.min() < 0.5 and c1.max() > 3.5 and c1.std() > 0.8, f"平均回数1：{c1.min():.1f}〜{c1.max():.1f} mA に広がる（0〜約 4 mA）")
ok(abs(c128.mean() - 2.0) < 0.1 and c128.nunique() <= 3, f"平均回数128：約 2 mA の {c128.nunique()} 本の棒（1〜2本）")

## 7. BME280 と Pico の内蔵温度センサの比較（`analyze_env.py`）

`logger.py` の `MODE = "env"`（1秒ごと）で2時間記録したつもりの CSV を作ります。部屋の温度は 23 °C から 2.5 °C ゆっくり上がり、空調で 20 分周期に ±0.8 °C 揺れるとしました。Pico の内蔵センサは、部屋の温度に**時定数 120 秒の1次遅れ**で追いつき、チップの発熱で **3.0 °C 高く**、1回ごとに 0.7 °C ばらつくとしました。

In [ ]:
syn.env("intro3/data/env.csv")
print(syn.TRUTH["env"])

In [ ]:
%%writefile intro3/analyze_env.py
#!/usr/bin/env python3
"""入門3：BME280 の温度と Pico の内蔵温度センサを回帰で比べる

使い方: cd ~/home-lab/intro3 && uv run python analyze_env.py data/env.csv

やること
  1. 2つの温度の時系列を描く
  2. pico_c = a + b·temp_c を最小二乗で当てはめる（傾き b と切片 a、その95%区間）
  3. 残差の自己相関（ダービン・ワトソン比）を調べ、HAC（Newey–West）の区間と比べる
  4. 変化（1つ前との差）どうしの相互相関から、どちらが何秒遅れて反応するかを調べる
"""
import sys

import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=FutureWarning)   # statsmodels の将来の仕様変更の予告を表示しない


def main(path):
    d = pd.read_csv(path)
    d["t_min"] = (d["host_time"] - d["host_time"].iloc[0]) / 60
    n = len(d)
    print(f"{n} 点、{d['t_min'].iloc[-1]:.1f} 分")
    print(f"BME280 の温度：{d['temp_c'].min():.2f}〜{d['temp_c'].max():.2f} °C（変化の幅 {np.ptp(d['temp_c']):.2f} °C）")
    if np.ptp(d["temp_c"]) < 1.0:
        print("  温度の変化が1 °C 未満なので、傾きの推定は不安定になる。部屋の温度が変わる時間帯に長く記録する")
    if "hum_pct" in d and d["hum_pct"].notna().any():
        print(f"湿度：{d['hum_pct'].min():.1f}〜{d['hum_pct'].max():.1f} %RH")
    else:
        print("湿度：なし（BMP280）")

    # 相互相関（1つ前との差どうし）。正のずれ = Pico の温度が遅れて変わる
    dt = float(np.median(np.diff(d["host_time"])))
    k = max(5, int(600 / dt))
    W = max(5, int(120 / dt))        # 内蔵センサの雑音が大きいので、約2分の移動平均でならしてから差をとる
    x1 = np.diff(d["temp_c"].rolling(W, center=True).mean().bfill().ffill().to_numpy())
    x2 = np.diff(d["pico_c"].rolling(W, center=True).mean().bfill().ffill().to_numpy())
    x1 = (x1 - x1.mean()) / x1.std()
    x2 = (x2 - x2.mean()) / x2.std()
    lags = np.arange(-k, k + 1)
    cc = [np.mean(x1[max(0, -L):len(x1) - max(0, L)] * x2[max(0, L):len(x2) - max(0, -L)]) for L in lags]
    best = int(lags[int(np.argmax(cc))])
    print(f"\n変化どうしの相互相関が最大になるずれ：{best * dt:+.0f} 秒（正なら Pico の温度が遅れて変わる）、"
          f"相関の最大値 {max(cc):.2f}")
    if max(cc) < 0.2:
        print("  相関が小さいので、ずれの推定は当てにならない（温度の変化が小さい・記録が短い）。ずらさない結果を使う")
        best = 0

    y_all = d["pico_c"].to_numpy()
    x_all = d["temp_c"].to_numpy()
    for title, sh in [("そのまま", 0), (f"BME280 を {best * dt:+.0f} 秒ずらして", best)]:
        if sh > 0:
            x, y = x_all[:-sh], y_all[sh:]
        elif sh < 0:
            x, y = x_all[-sh:], y_all[:sh]
        else:
            x, y = x_all, y_all
        X = sm.add_constant(x)
        ols = sm.OLS(y, X).fit()
        a, b = ols.params
        blo, bhi = ols.conf_int()[1]
        print(f"\n[{title}] pico_c = {a:.2f} + {b:.3f} × temp_c（{len(y)} 点）")
        print(f"  傾き b の95%区間：独立とみなすと {blo:.3f}〜{bhi:.3f}")
        for L in (10, int(np.sqrt(len(y))), max(len(y) // 20, 1)):
            hac = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": L})
            hlo, hhi = hac.conf_int()[1]
            print(f"                    HAC（ラグ {L:4d} 個 = {L * dt:5.0f} 秒）だと {hlo:.3f}〜{hhi:.3f}")
        print(f"  ダービン・ワトソン比 = {durbin_watson(ols.resid):.2f}（2 なら残差に自己相関なし。0 に近いほど正の自己相関が強い）"
              f"、残差の標準偏差 {ols.resid.std(ddof=2):.2f} °C")
        if sh == 0:
            a0, b0 = a, b
    print(f"\nずれ（Pico − BME280）の平均 {np.mean(y_all - x_all):.2f} °C（Pico のチップ自身の発熱などによる）")
    print("・HAC の区間はラグを増やすと広がり、どこかで落ち着く。落ち着いたところの幅を目安にする。")
    print("・独立とみなした区間は、自己相関が強いと狭すぎる（点の数ほどの情報はない）。")

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    ax[0].plot(d["t_min"], d["temp_c"], label="BME280")
    ax[0].plot(d["t_min"], d["pico_c"], label="Pico 内蔵", alpha=0.7)
    ax[0].set_xlabel("時間 [分]")
    ax[0].set_ylabel("温度 [°C]")
    ax[0].legend()
    ax[1].plot(x_all, y_all, ".", ms=2, alpha=0.5)
    xx = np.linspace(d["temp_c"].min(), d["temp_c"].max(), 50)
    ax[1].plot(xx, a0 + b0 * xx, "r-", label=f"直線 b={b0:.2f}（ずらさない）")
    ax[1].plot(xx, xx, "k--", lw=1, label="y = x")
    ax[1].set_xlabel("BME280 [°C]")
    ax[1].set_ylabel("Pico 内蔵 [°C]")
    ax[1].legend()
    ax[2].plot(lags * dt, cc)
    ax[2].axvline(best * dt, color="r", ls="--")
    ax[2].set_xlabel("ずれ [秒]（正 = Pico が遅れる）")
    ax[2].set_title("変化どうしの相互相関")
    for a_ in ax:
        a_.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig("env.png", dpi=130)
    print("env.png を保存しました")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit("使い方: uv run python analyze_env.py data/env.csv")
    main(sys.argv[1])

In [ ]:
out_e, rc = run("analyze_env.py data/env.csv", cwd="intro3")
show("intro3/env.png", width=900)

In [ ]:
# 確かめ（入門3-7 の温度の比較の結果の例）
ok(rc == 0, "analyze_env.py がエラーなく終わった")
diff = num(r"ずれ（Pico − BME280）の平均 ([\d.]+) °C", out_e)
ok(2 < diff < 4, f"Pico の内蔵センサは BME280 より {diff} °C 高い（数 °C。合成の真の値 3.0 °C）")
sd_res = num(r"残差の標準偏差 ([\d.]+) °C", out_e)
ok(0.5 <= sd_res <= 1.0, f"1回ごとのばらつき（残差の標準偏差）{sd_res} °C（約 0.5〜1 °C）")
lag = num(r"最大になるずれ：([+-]?\d+) 秒", out_e)
ok(20 <= lag <= 300 and "当てにならない" not in out_e, f"Pico の温度が {lag:+.0f} 秒遅れて変わる（数十秒〜数分。時定数 120 秒の1次遅れ）")
b = num(r"ずらして\] pico_c = [\d.]+ \+ ([\d.]+) ×", out_e)
ok(0.7 < b < 1.3, f"傾き b = {b}（1前後）")
blk = out_e.split("[そのまま]")[1].split("ずらして]")[0]   # ずらさない当てはめ（残差に自己相関が残る）
nlo, nhi = num(r"独立とみなすと ([\d.]+)〜", blk), num(r"独立とみなすと [\d.]+〜([\d.]+)", blk)
hlo, hhi = [float(v) for v in re.findall(r"だと ([\d.]+)〜([\d.]+)", blk)[-1]]
dw = num(r"ダービン・ワトソン比 = ([\d.]+)", blk)
ok(dw < 2 and hhi - hlo > 1.2 * (nhi - nlo), f"ずらさない当てはめ：DW = {dw}（2 より小さい）、HAC（いちばん長いラグ）の区間 {hlo}〜{hhi} は、独立とみなした {nlo}〜{nhi} より広い")

## 8. ロジックアナライザの記録からパルス幅と周期（入門4 手順11、`la_pulse.py`）

`servo_tool.py seq S1` の間に Pi 5 の `sigrok-cli --config samplerate=1m --samples 4000000 -O binary` で記録したつもりの `data/servo.bin`（1 MHz・4 秒、1サンプル1バイト、ビット0 = D0）を作ります。Pico 2 W の PWM（50 Hz）の1カウントは、150 MHz を 46 で割った約 0.307 µs として、指令のパルス幅をその刻みに丸めました（`duty_ns` の実際の丸め方は要確認）。Pico とロジアナの時計の差は +30 ppm、D1 には別の信号を入れて、ビット0 だけを取り出せるかも見ます。

In [ ]:
syn.servo_bin("intro4/data/servo.bin")
print(syn.TRUTH["servo_bin"], os.path.getsize("intro4/data/servo.bin"), "バイト")

In [ ]:
%%writefile intro4/la_pulse.py
#!/usr/bin/env python3
"""入門4：sigrok-cli の binary 出力から、D0（CH0）のパルス幅と周期を求める

使い方: cd ~/home-lab/intro4 && uv run python la_pulse.py data/servo.bin 1000000
  2つ目の引数はサンプリング周波数 [Hz]（sigrok-cli の samplerate と同じにする）
  8ch すべてを記録した場合、1サンプル = 1バイトで、ビット0 が D0
"""
import sys

import numpy as np

if len(sys.argv) < 3:
    sys.exit("使い方: uv run python la_pulse.py data/servo.bin 1000000")
fs = float(sys.argv[2])
raw = np.fromfile(sys.argv[1], dtype=np.uint8)
b = (raw & 0x01).astype(np.int8)
rise = np.flatnonzero((b[1:] == 1) & (b[:-1] == 0)) + 1
fall = np.flatnonzero((b[1:] == 0) & (b[:-1] == 1)) + 1
if len(rise) < 3:
    sys.exit("立ち上がりが少なすぎます。配線・GND・サンプルレート、servo_tool.py seq を実行中か確認してください")
fall = fall[fall > rise[0]]
n = min(len(rise), len(fall))
width_us = (fall[:n] - rise[:n]) / fs * 1e6
period_ms = np.diff(rise) / fs * 1e3
print(f"記録 {len(b)/fs:.2f} 秒、パルス {n} 個、1サンプル = {1e6/fs:.2f} µs")
print(f"周期：中央値 {np.median(period_ms):.3f} ms（{1000/np.median(period_ms):.2f} Hz）")
# 幅を10µs 単位で丸めてまとまりを作る（1000・1500・2000µs など）
groups = np.round(width_us, -1)
print("パルス幅のまとまり（10µs 単位で丸めたもの）:")
for g in np.unique(groups):
    w = width_us[groups == g]
    if len(w) < 3:
        continue
    print(f"  約{g:6.0f} µs：{len(w):4d} 個、平均 {w.mean():8.2f} µs、標準偏差 {w.std(ddof=1):.2f} µs、"
          f"最小 {w.min():.1f}、最大 {w.max():.1f}")
print("指令した値（1000・1500・2000µs）との差が、Pico の PWM の刻み＋1サンプル程度なら正常")

In [ ]:
out_la, rc = run("la_pulse.py data/servo.bin 1000000", cwd="intro4")
# 確かめ（入門4-7 のロジックアナライザの結果の例）
ok(rc == 0, "la_pulse.py がエラーなく終わった")
ok(abs(num(r"中央値 ([\d.]+) ms", out_la) - 20.0) < 0.005, "周期 20.00 ms 前後")
for cmd in (1000, 1500, 2000):
    m_ = re.search(rf"約 +{cmd} µs： +(\d+) 個、平均 +([\d.]+) µs、標準偏差 ([\d.]+) µs", out_la)
    ok(m_ and abs(float(m_.group(2)) - cmd) < 3 and float(m_.group(3)) <= 1.0,
       f"{cmd} µs：平均 {m_.group(2)} µs（指令値と数 µs 以内）、標準偏差 {m_.group(3)} µs（1サンプル程度）")

## 9. サーボの校正とヒステリシス（入門4 手順14〜16、`servo_analysis.py`）

まず本文 入門4-3 の数値例（電圧の落ち込み、470 µF のコンデンサ）を計算します。

In [ ]:
drop = 0.5 * (0.12 + 0.07 + 0.15 + 0.1 + 0.15)   # 電池・ヒューズ FU1・ポリスイッチ・シャント・配線
dV = 0.5 * 0.1e-3 / 470e-6
print(f"0.5 A で {drop:.2f} V 下がる、0.5 A を 0.1 ms 取り出すと ΔV = {dV:.3f} V、1 ms なら {dV*10:.2f} V")
ok(abs(drop - 0.3) < 0.01, "合計 約 0.6 Ω（0.59 Ω）に 0.5 A で約 0.3 V 下がる")
ok(abs(dV - 0.11) < 0.005 and dV * 10 > 1, "ΔV = IΔt/C ≈ 0.11 V、1 ms 続けば 1 V 以上")

`servo_tool.py calib` が保存する `data/calib_S?.csv`（列 `servo,rep,direction,pulse_us,angle_deg`）と同じ形のものを、サーボ2個分作ります。1000〜2000 µs を 100 µs おきに3回くり返し、`servo_tool.py` と同じく、行き（up）の 1000 µs と帰り（down）の 2000 µs は「手前から近づけない」ので入れません。真の値は S1 が 9.0°/100 µs・1500 µs で 92°・$h = -2.0°$、S2 が 8.6°/100 µs・89°・$h = -1.4°$ で、分度器の読み取りの誤差 0.4° と 0.5° 刻みの丸めを足しました。

In [ ]:
syn.calib("intro4/data/calib_S1.csv", "S1", b=0.090, a90=92.0, h=-2.0, seed=9)
syn.calib("intro4/data/calib_S2.csv", "S2", b=0.086, a90=89.0, h=-1.4, seed=10)
print(pd.read_csv("intro4/data/calib_S1.csv").head(12).to_string(index=False))

In [ ]:
%%writefile intro4/servo_analysis.py
#!/usr/bin/env python3
"""入門4：パルス幅と角度の校正（回帰）と、行きと帰りの差（ヒステリシス）の推定

使い方: cd ~/home-lab/intro4 && uv run python servo_analysis.py data/calib_S1.csv data/calib_S2.csv

やること
  1. サーボごとに 角度 = a + b·パルス幅 を当てはめる（1µs あたり何度か）
  2. 角度 = a + b·パルス幅 + h·向き（up = +1/2、down = −1/2）で、行きと帰りの差 h を推定する
  3. 2次の項を足したモデルと AIC で比べる（直線で十分か）
  4. 欲しい角度にするパルス幅（逆推定）とその95%区間
  5. サーボが2個あれば、傾きがサーボで違うかを分散分析（F 検定）で調べる
"""
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", category=FutureWarning)

TARGETS = [45, 90, 135]      # 逆推定したい角度 [°]


def inverse(fit, angle):
    """角度 = a + b·x を x について解き、デルタ法で標準誤差を出す（向きの効果は平均の0とする）"""
    a, b = fit.params["Intercept"], fit.params["x"]
    cov = fit.cov_params().loc[["Intercept", "x"], ["Intercept", "x"]].to_numpy()
    x = (angle - a) / b
    g = np.array([-1 / b, -(angle - a) / b ** 2])     # x を a・b で微分したもの
    se = float(np.sqrt(g @ cov @ g))
    return x, se


def main(paths):
    if not paths:
        sys.exit("使い方: uv run python servo_analysis.py data/calib_S1.csv [data/calib_S2.csv]")
    d = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True).dropna()
    d["x"] = d["pulse_us"] - 1500.0                     # 1500µs を0にする（切片 = 1500µs のときの角度）
    d["dir"] = np.where(d["direction"] == "up", 0.5, -0.5)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    for sv, g in d.groupby("servo"):
        print(f"\n===== {sv}（{len(g)} 点、{g['rep'].nunique()} 回くり返し）=====")
        m1 = smf.ols("angle_deg ~ x", g).fit()
        m2 = smf.ols("angle_deg ~ x + dir", g).fit()
        m3 = smf.ols("angle_deg ~ x + I(x**2) + dir", g).fit()
        b = m2.params["x"]
        blo, bhi = m2.conf_int().loc["x"]
        h = m2.params["dir"]
        hlo, hhi = m2.conf_int().loc["dir"]
        print(f"傾き b = {b*100:.2f} °/100µs（95%区間 {blo*100:.2f}〜{bhi*100:.2f}）、"
              f"1500µs のときの角度 {m2.params['Intercept']:.1f}°")
        print(f"行き − 帰り h = {h:+.2f}°（95%区間 {hlo:+.2f}〜{hhi:+.2f}）"
              f" → パルス幅に直すと {h/b:+.1f} µs（不感帯・ガタの目安）")
        print(f"残差の標準偏差：向きなし {np.sqrt(m1.scale):.2f}°、向きあり {np.sqrt(m2.scale):.2f}°")
        print(f"AIC：直線＋向き {m2.aic:.1f}、2次＋向き {m3.aic:.1f}（小さいほうがよい。差が2未満ならほぼ同じ）")
        for ang in TARGETS:
            x, se = inverse(m2, ang)
            us = x + 1500
            if d["pulse_us"].min() - 100 <= us <= d["pulse_us"].max() + 100:
                print(f"  {ang:3d}° にするパルス幅：{us:.0f} µs（±{1.96*se:.1f} µs、95%）")
            else:
                print(f"  {ang:3d}°：測った範囲の外なので推定しない（外挿になる）")
        for dirn, mk in (("up", "^"), ("down", "v")):
            gg = g[g["direction"] == dirn]
            ax[0].plot(gg["pulse_us"], gg["angle_deg"], mk, ms=5, alpha=0.6, label=f"{sv} {dirn}")
            ax[1].plot(gg["pulse_us"], m1.resid[gg.index], mk, ms=5, alpha=0.6, label=f"{sv} {dirn}")
        xx = np.linspace(g["x"].min(), g["x"].max(), 50)
        ax[0].plot(xx + 1500, m2.params["Intercept"] + b * xx, "-", lw=1)

    if d["servo"].nunique() >= 2:
        full = smf.ols("angle_deg ~ x * C(servo) + dir", d).fit()
        red = smf.ols("angle_deg ~ x + C(servo) + dir", d).fit()
        t = anova_lm(red, full)
        print("\n===== サーボによって傾きが違うか（x × サーボ の交互作用）=====")
        print(f"F = {t['F'].iloc[1]:.2f}、p = {t['Pr(>F)'].iloc[1]:.3g}")
        print("・p が小さければ、サーボごとに校正が必要。")
        print("・ただし、サーボが2個だけなので、「MG90D という型番全体」の傾きのばらつきは推定できない（独立な単位は2個）。")
    ax[0].set_xlabel("パルス幅 [µs]")
    ax[0].set_ylabel("角度 [°]")
    ax[0].set_title("パルス幅と角度（線は向きを平均した直線）")
    ax[1].axhline(0, color="k", lw=0.8)
    ax[1].set_xlabel("パルス幅 [µs]")
    ax[1].set_ylabel("残差 [°]")
    ax[1].set_title("向きを入れない直線の残差（▲行きと▼帰りの差がヒステリシス）")
    for a_ in ax:
        a_.grid(True, alpha=0.3)
        a_.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig("calib.png", dpi=130)
    print("\ncalib.png を保存しました")


if __name__ == "__main__":
    main(sys.argv[1:])

In [ ]:
out_s, rc = run("servo_analysis.py data/calib_S1.csv data/calib_S2.csv", cwd="intro4")
show("intro4/calib.png", width=900)

In [ ]:
# 確かめ（入門4-7 の校正・逆推定の結果の例）
ok(rc == 0, "servo_analysis.py がエラーなく終わった")
for sv in ("S1", "S2"):
    blk = out_s.split(f"===== {sv}")[1].split("=====")[1]
    t = syn.TRUTH[f"calib_{sv}"]
    b, blo, bhi = [num(p, blk) for p in (r"傾き b = ([\d.]+)", r"95%区間 ([\d.]+)〜", r"95%区間 [\d.]+〜([\d.]+)）")]
    ok(7 < b < 11 and blo - 0.02 <= t["b"] * 100 <= bhi + 0.02, f"{sv}：傾き {b} °/100µs（約 9°/100µs）、区間 {blo}〜{bhi} が真の値 {t['b']*100:.1f} を含む")
    h = num(r"h = ([+-][\d.]+)°", blk)
    hlo, hhi = num(r"h = [+-][\d.]+°（95%区間 ([+-][\d.]+)〜", blk), num(r"h = [+-][\d.]+°（95%区間 [+-][\d.]+〜([+-][\d.]+)）", blk)
    ok(h < 0 and hhi < 0 and 1 <= abs(h) <= 5, f"{sv}：h = {h}°（負、|h| は 1〜数度）、区間 {hlo}〜{hhi} は 0 をまたがない")
    s1, s2 = num(r"向きなし ([\d.]+)°", blk), num(r"向きあり ([\d.]+)°", blk)
    ok(s2 < s1 and 0.3 <= s2 <= 1.0, f"{sv}：残差の標準偏差は向きを入れると {s1}° → {s2}°（0.5〜1° 程度）")
    pm = num(r" 90° にするパルス幅：\d+ µs（±([\d.]+) µs", blk)
    ok(pm < 10, f"{sv}：90° にするパルス幅の区間は ±{pm} µs（±数 µs と狭い）")
ok(num(r"p = ([\d.e+-]+)", out_s.split("交互作用")[1]) < 0.05, "傾きの差（0.4°/100µs）は F 検定で有意（本文：有意になることも、ならないこともある）")

## おわりに

- 入門1〜4 の Mac 側の解析ファイル（`led_iv.py`、`bounce_analysis.py`、`joint_analysis.py`、`test_bme280_vector.py`、`analyze_noise.py`、`analyze_env.py`、`la_pulse.py`、`servo_analysis.py`）を、真の値のわかった合成データで動かし、本文の「結果の例」の目安と合うことを確かめました。
- 統計の考え方（分位点の区間、疑似反復、頑健な外れ値の判定とマスキング、ICC、自己相関と HAC・有効サンプル数、アラン偏差、デルタ法での逆推定）は、第1章からあとの章でもくり返し使います。
- 3 節の「99%点の上限」は、両側 95% の区間なので、上の端が決まるのは 368 件からでした（299 件は片側の場合）。
- 自分の測定値で試すときは、記録用紙の CSV やロジアナの `.bin` を、このノートブックの `intro1/`〜`intro4/` に置いて、同じ `run(...)` を実行してください（執筆時には実物のデータでは確かめていません）。